# Meta-Brain V3 — conectoma completo treinável no Colab Pro

Escolha **Ambiente de execução → Alterar tipo de ambiente → GPU**. Ative **RAM alta** se disponível. Execute as células em ordem. Não é necessário clonar o GitHub: o código está embutido neste notebook.

Esta versão treina a visão, ganhos/constantes por tipo de neurônio e a decisão motora com PPO recorrente. Todas as ações passam pelo conectoma completo `Traced`; não há atalho da imagem para a política. A topologia e os sinais sinápticos permanecem fixos. A V2 continua sendo a referência com reservatório congelado.

**Persistência:** autorize o Drive na célula 2. Dados/cache são reutilizados; `latest.pt` permite continuar o treinamento em outra sessão. Use o mesmo `RUN_NAME`, `VISION` e `SEED` e execute tudo novamente. A retomada restaura pesos, otimizador e contadores; os episódios em andamento recomeçam. Não é reprodução bit a bit de uma sessão ininterrupta.

**Limites:** Colab Pro não garante uma GPU específica nem duração fixa. O orçamento abaixo limita a sessão de treinamento e salva antes da avaliação final; download, preparação e avaliação levam tempo adicional. Nenhum resultado de vitória está pré-calculado. O sucesso só é marcado se todos os episódios finais determinísticos alcançarem a bandeira sem morrer.

Fonte: [limites oficiais do Colab](https://research.google.com/colaboratory/faq.html).

## 1. Instalação

In [ ]:
# Não reinstala PyTorch nem faz downgrade de NumPy do Colab.
# Pacotes modernos usam Gymnasium e funcionam com NumPy 2.
%pip -q install gym-super-mario-bros==9.1.0 nes-py==9.0.1 'gymnasium>=1.0,<2' pyarrow requests
import shutil, subprocess
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)


## 2. Drive e parâmetros

In [ ]:
import os, sys, json, gc
from pathlib import Path
import torch

USE_DRIVE = True  # @param {type:"boolean"}
RUN_NAME = 'mario_v3'  # @param {type:"string"}
VISION = 'real'  # @param ["real", "zero", "clock"]
SEED = 123  # @param {type:"integer"}
TARGET_UPDATES = 10000  # @param {type:"integer"}
SESSION_HOURS = 3.0  # @param {type:"number"}
FINAL_EPISODES = 20  # @param {type:"integer"}
N_ENVS = 4  # @param {type:"integer"}
BPTT = 16  # @param {type:"integer"}
MICRO_STEPS = 4  # @param {type:"integer"}

# Overrides for automated offline smoke tests only.
OFFLINE = os.environ.get('MB_OFFLINE') == '1'
DEVICE = os.environ.get('MB_DEVICE', 'cuda')
if DEVICE != 'cuda' and not OFFLINE:
    raise RuntimeError('O experimento completo exige GPU CUDA. Selecione GPU no menu do Colab.')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('GPU não está ativa. Altere o tipo de ambiente e execute novamente.')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME deve ser apenas um nome de pasta, sem barras.')
if not USE_DRIVE and not OFFLINE:
    print('USE_DRIVE=False: resultados só durarão enquanto esta sessão existir.')
ROOT = Path('/content/meta_brain_v3')
if USE_DRIVE and not OFFLINE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Meta-Brain')
RUN_DIR = Path(os.environ.get('MB_OUT_DIR', str(ROOT / 'runs' / f'{RUN_NAME}_{VISION}_seed{SEED}')))
DATA_DIR = Path(os.environ.get('MB_DATA_DIR', '/content/malecns_v1_0'))
CACHE_DIR = ROOT / 'graph_cache'
PERSISTENT_DATA = ROOT / 'data' if USE_DRIVE and not OFFLINE else None
if OFFLINE:
    CACHE_DIR = DATA_DIR / 'cache'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR = Path(os.environ.get('MB_V3_RUNTIME', '/content/meta_brain_v3_runtime'))
print('Resultados e checkpoints:', RUN_DIR)


## 3. Código autocontido

In [ ]:
# Exact package sources; regenerated, not downloaded from a mutable branch.
SOURCE_DIGEST = '5438430f4b671de8d6d19f00f18050286aa4aa4bf3bfd5974596e18ee45fa892'
SOURCE_FILES = {'meta_brain/graph.py': '"""Vectorized MaleCNS connectome loader with an on-disk cache (milestone M0).\n\nThe MaleCNS v1.0 "flat connectome" release ships three files (the ones\n``scripts/download_data.py`` fetches):\n\n* ``body-annotations-male-cns-v1.0-minconf-0.5.feather``: one row per body\n  (``bodyId``, ``status``, ``type``, ``superclass``, ...);\n* ``body-neurotransmitters-male-cns-v1.0.feather``: per-body NT predictions\n  (``body``, ``consensus_nt``, ``predicted_nt``);\n* ``connectome-weights-male-cns-v1.0-minconf-0.5.feather``: ~152M weighted\n  rows (``body_pre``, ``body_post``, ``weight``), one per connected body pair,\n  including the many untraced fragments. Keeping ``Traced`` bodies leaves\n  ~25.6M edges. Repeated pairs (none in v1.0) would be summed.\n\nThe legacy layout (``*meta*.feather`` + ``*simple*edgelist*.feather``) is still\naccepted.\n\nThe weights table is streamed one Arrow record batch at a time, body ids are\nremapped with ``searchsorted``, and duplicate pairs are merged incrementally so\npeak memory stays bounded by the number of unique pairs rather than the number\nof rows. Every row is accounted for in :class:`LoadReport`.\n\nThe resulting :class:`ConnectomeArrays` stores edges sorted by ``(post, pre)``,\nwhich is CSR row order for :mod:`meta_brain.sparse_ops`.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, Iterator, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nLOADER_VERSION = 2\n\n# Candidate column names, in priority order. The first one present is used.\nMETA_COLUMNS: Dict[str, Tuple[str, ...]] = {\n    "body_id": ("bodyId", "body_id", "body", "root_id"),\n    "status": ("status",),\n    "type": ("type", "cell_type"),\n    "superclass": ("superclass", "super_class"),\n    "class": ("class", "cell_class"),\n    "nt": ("predictedNt", "consensusNt", "nt_type", "predicted_nt"),\n    "nt_confidence": ("predictedNtConfidence", "predictedNtProb", "nt_type_score"),\n    "side": ("somaSide", "rootSide", "side"),\n}\nNT_FILE_COLUMNS: Dict[str, Tuple[str, ...]] = {\n    "body_id": ("body", "bodyId", "body_id"),\n    "consensus": ("consensus_nt", "consensusNt"),\n    "predicted": ("predicted_nt", "predictedNt"),\n    "confidence": ("predicted_nt_prob", "predicted_nt_confidence", "predictedNtProb",\n                   "predictedNtConfidence"),\n}\nEDGE_COLUMNS: Dict[str, Tuple[str, ...]] = {\n    "pre": ("body_pre", "bodyId_pre", "pre", "pre_root_id"),\n    "post": ("body_post", "bodyId_post", "post", "post_root_id"),\n    "weight": ("weight", "syn_count", "count"),\n}\n\nFLAT_PATTERNS = {\n    "annotations": "body-annotations-*.feather",\n    "neurotransmitters": "body-neurotransmitters-*.feather",\n    "edges": "connectome-weights-*.feather",\n}\nLEGACY_PATTERNS = {"annotations": "*meta*.feather", "edges": "*simple*edgelist*.feather"}\n\nDEFAULT_STATUSES: Tuple[str, ...] = ("Traced",)\nDEFAULT_MERGE_EVERY = 30_000_000  # rows accumulated before an incremental merge\n\n# Neurotransmitter codes stored per neuron. Signs are a separate, overridable\n# mapping so downstream layers (e.g. dopamine as a teaching signal) can decide\n# how to treat modulatory transmitters.\nNT_CODES: Dict[str, int] = {\n    "unknown": 0,\n    "acetylcholine": 1,\n    "gaba": 2,\n    "glutamate": 3,\n    "histamine": 4,\n    "dopamine": 5,\n    "serotonin": 6,\n    "octopamine": 7,\n    "tyramine": 8,\n}\nNT_ALIASES: Dict[str, str] = {\n    "ach": "acetylcholine",\n    "glu": "glutamate",\n    "da": "dopamine",\n    "5ht": "serotonin",\n    "ser": "serotonin",\n    "oa": "octopamine",\n    "ta": "tyramine",\n    "his": "histamine",\n}\n# Fast-transmission sign by NT code. Glutamate is inhibitory at most fly\n# central synapses (GluCl) and histamine is the inhibitory photoreceptor\n# transmitter. Modulatory and unknown transmitters default to +1; layer 2\n# handles dopamine separately through ``nt_code``.\nDEFAULT_NT_SIGN: Dict[str, float] = {\n    "unknown": 1.0,\n    "acetylcholine": 1.0,\n    "gaba": -1.0,\n    "glutamate": -1.0,\n    "histamine": -1.0,\n    "dopamine": 1.0,\n    "serotonin": 1.0,\n    "octopamine": 1.0,\n    "tyramine": 1.0,\n}\n\nSENSORY_SUPERCLASSES = ("ol_sensory", "cb_sensory", "vnc_sensory", "sensory_ascending", "olsn")\nMOTOR_SUPERCLASSES = ("descending_neuron", "vnc_motor", "cb_motor", "dn", "motor")\nDESCENDING_SUPERCLASSES = ("descending_neuron", "dn")\n\n\n@dataclass\nclass DataFiles:\n    annotations: Path\n    edges: Path\n    neurotransmitters: Optional[Path] = None\n    layout: str = "flat"\n\n    def all(self) -> List[Path]:\n        return [p for p in (self.annotations, self.neurotransmitters, self.edges) if p is not None]\n\n\n@dataclass\nclass LoadReport:\n    """Every body and edge row that was read, dropped or merged, so counts can be audited."""\n\n    layout: str = ""\n    files: Dict[str, str] = field(default_factory=dict)\n    columns_used: Dict[str, str] = field(default_factory=dict)\n    statuses: List[str] = field(default_factory=list)\n    bodies_annotated: int = 0\n    dropped_by_status: int = 0\n    n_neurons: int = 0\n    nt_source: str = ""\n    edge_rows_read: int = 0\n    dropped_unknown_pre: int = 0\n    dropped_unknown_post: int = 0\n    dropped_self_loops: int = 0\n    duplicate_rows_merged: int = 0\n    dropped_below_min_weight: int = 0\n    min_weight: float = 1.0\n    edges_retained: int = 0\n    synapses_retained: int = 0\n    n_cell_types: int = 0\n    n_untyped_neurons: int = 0\n    n_type_pairs: int = 0\n    nt_counts: Dict[str, int] = field(default_factory=dict)\n    build_seconds: float = 0.0\n    load_seconds: float = 0.0\n    from_cache: bool = False\n\n    def accounted_rows(self) -> int:\n        return (self.edges_retained + self.dropped_unknown_pre + self.dropped_unknown_post\n                + self.dropped_self_loops + self.duplicate_rows_merged + self.dropped_below_min_weight)\n\n    def summary(self) -> str:\n        status = f" status in {self.statuses}" if self.statuses else ""\n        lines = [\n            f"layout={self.layout}  bodies annotated={self.bodies_annotated:,}"\n            f"  dropped by{status or \' status\'}={self.dropped_by_status:,}",\n            f"neurons={self.n_neurons:,}  NT source={self.nt_source}",\n            f"edge rows read={self.edge_rows_read:,}",\n            f"  dropped unknown pre={self.dropped_unknown_pre:,} post={self.dropped_unknown_post:,}",\n            f"  dropped self-loops={self.dropped_self_loops:,}",\n            f"  duplicate rows merged={self.duplicate_rows_merged:,}",\n            f"  pairs below min weight {self.min_weight:g}={self.dropped_below_min_weight:,}",\n            f"edges retained={self.edges_retained:,} (synapses={self.synapses_retained:,})",\n            f"cell types={self.n_cell_types:,} (untyped neurons={self.n_untyped_neurons:,}), "\n            f"type pairs={self.n_type_pairs:,}",\n            f"build time={self.build_seconds:.1f}s"\n            + (f", cache load time={self.load_seconds:.1f}s" if self.from_cache else ""),\n        ]\n        return "\\n".join(lines)\n\n\n@dataclass\nclass ConnectomeArrays:\n    """Canonical sparse connectome. Edges are sorted by ``(post, pre)``."""\n\n    body_id: np.ndarray  # (N,) int64\n    pre: np.ndarray  # (E,) int32, neuron index\n    post: np.ndarray  # (E,) int32, neuron index\n    count: np.ndarray  # (E,) float32, synapse count summed over rows\n    nt_code: np.ndarray  # (N,) int8, see NT_CODES\n    nt_confidence: np.ndarray  # (N,) float32, NaN when unavailable\n    cell_type: np.ndarray  # (N,) int32 index into type_names\n    type_names: List[str]\n    superclass: np.ndarray  # (N,) int16 index into superclass_names\n    superclass_names: List[str]\n    side: np.ndarray  # (N,) int8: 0 unknown, 1 left, 2 right, 3 center\n    report: LoadReport = field(default_factory=LoadReport)\n\n    @property\n    def n(self) -> int:\n        return int(len(self.body_id))\n\n    @property\n    def e(self) -> int:\n        return int(len(self.pre))\n\n    @property\n    def n_types(self) -> int:\n        return len(self.type_names)\n\n    def sign(self, nt_sign: Optional[Dict[str, float]] = None) -> np.ndarray:\n        """Per-neuron fast-transmission sign from the NT prediction."""\n        table = dict(DEFAULT_NT_SIGN)\n        if nt_sign:\n            table.update(nt_sign)\n        lut = np.ones(len(NT_CODES), dtype=np.float32)\n        for name, code in NT_CODES.items():\n            lut[code] = table.get(name, 1.0)\n        return lut[self.nt_code]\n\n    def superclass_mask(self, names: Sequence[str]) -> np.ndarray:\n        wanted = {s.lower() for s in names}\n        codes = [i for i, s in enumerate(self.superclass_names) if s in wanted]\n        return np.isin(self.superclass, codes)\n\n    def sensory_indices(self) -> np.ndarray:\n        return np.flatnonzero(self.superclass_mask(SENSORY_SUPERCLASSES))\n\n    def motor_indices(self) -> np.ndarray:\n        return np.flatnonzero(self.superclass_mask(MOTOR_SUPERCLASSES))\n\n    def descending_indices(self) -> np.ndarray:\n        return np.flatnonzero(self.superclass_mask(DESCENDING_SUPERCLASSES))\n\n    def type_pair_index(self) -> Tuple[np.ndarray, np.ndarray]:\n        """Map every edge to a connected ``(pre_type, post_type)`` pair.\n\n        Returns ``(edge_pair, pair_types)`` where ``edge_pair`` is ``(E,)``\n        int32 and ``pair_types`` is ``(P, 2)`` int32 with columns\n        ``[pre_type, post_type]``.\n        """\n        return type_pair_index(self.pre, self.post, self.cell_type, self.n_types)\n\n    def csr_rowptr(self) -> np.ndarray:\n        """CSR row pointer with rows = postsynaptic neuron."""\n        counts = np.bincount(self.post, minlength=self.n)\n        rowptr = np.zeros(self.n + 1, dtype=np.int64)\n        np.cumsum(counts, out=rowptr[1:])\n        return rowptr\n\n    def replace_edges(self, pre: np.ndarray, post: np.ndarray, count: np.ndarray) -> "ConnectomeArrays":\n        """Copy with a different edge set (used by the shuffle controls)."""\n        pre, post, count = sort_edges(pre, post, count)\n        out = ConnectomeArrays(**{**self.__dict__, "pre": pre, "post": post, "count": count})\n        out.report = LoadReport(**{**asdict(self.report), "edges_retained": len(pre),\n                                   "synapses_retained": int(count.sum())})\n        return out\n\n\n# --------------------------------------------------------------------------- #\n# Helpers\n# --------------------------------------------------------------------------- #\n\ndef _pick(columns: Sequence[str], candidates: Sequence[str], required: bool, what: str) -> Optional[str]:\n    for c in candidates:\n        if c in columns:\n            return c\n    if required:\n        raise ValueError(f"{what}: none of {list(candidates)} found in columns {list(columns)}")\n    return None\n\n\ndef _nt_names(values) -> pd.Series:\n    names = pd.Series(values).fillna("").astype(str).str.strip().str.lower()\n    return names.map(lambda s: NT_ALIASES.get(s, s))\n\n\ndef _nt_codes(names: pd.Series) -> np.ndarray:\n    return names.map(lambda s: NT_CODES.get(s, 0)).to_numpy(dtype=np.int8)\n\n\ndef _normalize_side(values: Optional[pd.Series], n: int) -> np.ndarray:\n    if values is None:\n        return np.zeros(n, dtype=np.int8)\n    s = values.fillna("").astype(str).str.strip().str.lower().str[:1]\n    return s.map({"l": 1, "r": 2, "c": 3, "m": 3}).fillna(0).to_numpy(dtype=np.int8)\n\n\ndef cell_type_keys(meta: pd.DataFrame, type_col: Optional[str], superclass_col: Optional[str],\n                   class_col: Optional[str]) -> Tuple[np.ndarray, int]:\n    """Type key per neuron; untyped neurons fall back to ``untyped:<superclass>:<class>``."""\n    n = len(meta)\n\n    def col(name: Optional[str]) -> pd.Series:\n        if name is None:\n            return pd.Series([""] * n, index=meta.index)\n        return meta[name].fillna("").astype(str).str.strip()\n\n    typ, sup, cls = col(type_col), col(superclass_col), col(class_col)\n    untyped = (typ == "") | typ.str.lower().isin({"nan", "none", "unknown"})\n    keys = typ.where(~untyped, "untyped:" + sup.str.lower() + ":" + cls.str.lower())\n    return keys.to_numpy(dtype=object), int(untyped.sum())\n\n\ndef remap_ids(sorted_ids: np.ndarray, order: np.ndarray, raw: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    """Vectorized bodyId -> row index. Returns ``(index, valid_mask)``."""\n    if len(sorted_ids) == 0:\n        return np.zeros(len(raw), np.int32), np.zeros(len(raw), bool)\n    pos = np.searchsorted(sorted_ids, raw)\n    pos = np.minimum(pos, len(sorted_ids) - 1)\n    valid = sorted_ids[pos] == raw\n    return order[pos].astype(np.int32), valid\n\n\ndef sort_edges(pre: np.ndarray, post: np.ndarray, count: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    order = np.lexsort((pre, post))\n    return (pre[order].astype(np.int32, copy=False), post[order].astype(np.int32, copy=False),\n            count[order].astype(np.float32, copy=False))\n\n\ndef merge_duplicates(pre: np.ndarray, post: np.ndarray, count: np.ndarray, n: int\n                     ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:\n    """Sum the counts of repeated ``(pre, post)`` rows. Returns the merged edges."""\n    key = post.astype(np.int64) * n + pre.astype(np.int64)\n    uniq, inverse = np.unique(key, return_inverse=True)\n    merged = len(key) - len(uniq)\n    if merged == 0:\n        return (*sort_edges(pre, post, count), 0)\n    summed = np.bincount(inverse, weights=count.astype(np.float64), minlength=len(uniq))\n    return ((uniq % n).astype(np.int32), (uniq // n).astype(np.int32),\n            summed.astype(np.float32), int(merged))\n\n\ndef _merge_keys(keys: List[np.ndarray], weights: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:\n    """Merge ``post * n + pre`` keys, summing weights; output is sorted and unique."""\n    k = np.concatenate(keys)\n    w = np.concatenate(weights)\n    uniq, inverse = np.unique(k, return_inverse=True)\n    return uniq, np.bincount(inverse, weights=w, minlength=len(uniq))\n\n\ndef type_pair_index(pre: np.ndarray, post: np.ndarray, cell_type: np.ndarray, n_types: int\n                    ) -> Tuple[np.ndarray, np.ndarray]:\n    key = cell_type[pre].astype(np.int64) * n_types + cell_type[post].astype(np.int64)\n    uniq, inverse = np.unique(key, return_inverse=True)\n    pair_types = np.stack([uniq // n_types, uniq % n_types], axis=1).astype(np.int32)\n    return inverse.astype(np.int32), pair_types\n\n\ndef _feather_columns(path: Path) -> List[str]:\n    import pyarrow as pa\n    import pyarrow.ipc as ipc\n\n    try:\n        return list(ipc.open_file(pa.memory_map(str(path), "r")).schema.names)\n    except pa.ArrowInvalid:\n        return list(pd.read_feather(path).columns)\n\n\ndef _iter_edge_batches(path: Path, columns: List[str]) -> Iterator[Dict[str, np.ndarray]]:\n    """Yield the edge table one Arrow record batch at a time (bounded memory)."""\n    import pyarrow as pa\n    import pyarrow.ipc as ipc\n\n    try:\n        reader = ipc.open_file(pa.memory_map(str(path), "r"))\n    except pa.ArrowInvalid:\n        # Feather v1 or a non-IPC file: fall back to a single full read.\n        df = pd.read_feather(path, columns=columns)\n        yield {c: df[c].to_numpy() for c in columns}\n        return\n    for i in range(reader.num_record_batches):\n        batch = reader.get_batch(i)\n        yield {c: batch.column(c).to_numpy(zero_copy_only=False) for c in columns}\n\n\ndef find_data_files(data_dir: str | Path) -> DataFiles:\n    """Locate the flat-connectome release, falling back to the legacy layout."""\n    data_dir = Path(data_dir)\n\n    def first(pattern: str) -> Optional[Path]:\n        hits = sorted(data_dir.glob(pattern))\n        return hits[0] if hits else None\n\n    ann, edges = first(FLAT_PATTERNS["annotations"]), first(FLAT_PATTERNS["edges"])\n    if ann and edges:\n        return DataFiles(ann, edges, first(FLAT_PATTERNS["neurotransmitters"]), "flat")\n    ann, edges = first(LEGACY_PATTERNS["annotations"]), first(LEGACY_PATTERNS["edges"])\n    if ann and edges:\n        return DataFiles(ann, edges, None, "legacy")\n    raise FileNotFoundError(\n        f"No connectome in {data_dir}. Expected {FLAT_PATTERNS[\'annotations\']} + "\n        f"{FLAT_PATTERNS[\'edges\']} (+ {FLAT_PATTERNS[\'neurotransmitters\']}); "\n        "run scripts/download_data.py.")\n\n\ndef _join_neurotransmitters(nt_path: Path, body_id: np.ndarray, report: LoadReport\n                            ) -> Tuple[np.ndarray, np.ndarray]:\n    """Consensus NT where available and not \'unclear\', else the per-body prediction."""\n    cols = _feather_columns(nt_path)\n    used = {k: _pick(cols, cands, k in ("body_id",), f"neurotransmitters {k}")\n            for k, cands in NT_FILE_COLUMNS.items()}\n    if used["consensus"] is None and used["predicted"] is None:\n        raise ValueError(f"neurotransmitters: no consensus/predicted column in {cols}")\n    report.columns_used.update({f"nt.{k}": v for k, v in used.items() if v})\n    nt_df = pd.read_feather(nt_path, columns=[v for v in used.values() if v])\n    nt_df = nt_df.drop_duplicates(used["body_id"]).set_index(used["body_id"])\n\n    def column(key: str) -> pd.Series:\n        if used[key] is None:\n            return pd.Series([""] * len(body_id))\n        return _nt_names(nt_df[used[key]].reindex(body_id).to_numpy())\n\n    cons, pred = column("consensus"), column("predicted")\n    usable = (cons != "") & (cons != "unclear")\n    names = cons.where(usable, pred)\n    report.nt_source = "consensus_nt, else predicted_nt" if used["consensus"] else "predicted_nt"\n    conf = (pd.to_numeric(nt_df[used["confidence"]].reindex(body_id), errors="coerce")\n            .to_numpy(dtype=np.float32) if used["confidence"]\n            else np.full(len(body_id), np.nan, dtype=np.float32))\n    return _nt_codes(names), conf\n\n\n# --------------------------------------------------------------------------- #\n# Loading and caching\n# --------------------------------------------------------------------------- #\n\ndef build_connectome(files: DataFiles, min_weight: float = 1.0, drop_self_loops: bool = True,\n                     statuses: Optional[Sequence[str]] = DEFAULT_STATUSES, verbose: bool = True,\n                     merge_every: int = DEFAULT_MERGE_EVERY) -> ConnectomeArrays:\n    """Build :class:`ConnectomeArrays` from MaleCNS feather files.\n\n    ``statuses`` keeps only bodies whose ``status`` is listed (``None`` keeps\n    all). ``min_weight`` is applied to the *merged* pair weight, i.e. the\n    total synapse count between two neurons.\n    """\n    t0 = time.perf_counter()\n    report = LoadReport(layout=files.layout, min_weight=float(min_weight),\n                        files={k: str(v) for k, v in asdict(files).items() if isinstance(v, Path)})\n\n    meta = pd.read_feather(files.annotations)\n    cols = list(meta.columns)\n    used = {k: _pick(cols, cands, k == "body_id", f"annotations {k}") for k, cands in META_COLUMNS.items()}\n    report.columns_used.update({f"meta.{k}": v for k, v in used.items() if v})\n    report.bodies_annotated = len(meta)\n    if statuses and used["status"]:\n        keep = meta[used["status"]].astype(str).isin(set(statuses))\n        report.dropped_by_status = int((~keep).sum())\n        report.statuses = list(statuses)\n        meta = meta[keep.to_numpy()].reset_index(drop=True)\n\n    body_id = meta[used["body_id"]].astype(np.int64).to_numpy()\n    if len(np.unique(body_id)) != len(body_id):\n        raise ValueError("Neuron annotations contain duplicate body ids.")\n    n = len(body_id)\n    report.n_neurons = n\n\n    keys, n_untyped = cell_type_keys(meta, used["type"], used["superclass"], used["class"])\n    type_names, cell_type = np.unique(keys.astype(str), return_inverse=True)\n    report.n_cell_types, report.n_untyped_neurons = len(type_names), n_untyped\n\n    sup = (meta[used["superclass"]].fillna("").astype(str).str.strip().str.lower()\n           if used["superclass"] else pd.Series([""] * n))\n    superclass_names, superclass = np.unique(sup.to_numpy(dtype=str), return_inverse=True)\n\n    if files.neurotransmitters is not None:\n        nt_code, nt_conf = _join_neurotransmitters(files.neurotransmitters, body_id, report)\n    else:\n        nt_code = (_nt_codes(_nt_names(meta[used["nt"]].to_numpy())) if used["nt"]\n                   else np.zeros(n, dtype=np.int8))\n        nt_conf = (pd.to_numeric(meta[used["nt_confidence"]], errors="coerce").to_numpy(dtype=np.float32)\n                   if used["nt_confidence"] else np.full(n, np.nan, dtype=np.float32))\n        report.nt_source = f"annotations.{used[\'nt\']}" if used["nt"] else "none"\n    inv_nt = {v: k for k, v in NT_CODES.items()}\n    report.nt_counts = {inv_nt[int(c)]: int(k) for c, k in zip(*np.unique(nt_code, return_counts=True))}\n    side = _normalize_side(meta[used["side"]] if used["side"] else None, n)\n    del meta\n\n    order = np.argsort(body_id, kind="stable")\n    sorted_ids = body_id[order]\n\n    edge_cols = _feather_columns(files.edges)\n    ecol = {k: _pick(edge_cols, cands, k != "weight", f"edges {k}") for k, cands in EDGE_COLUMNS.items()}\n    report.columns_used.update({f"edge.{k}": v for k, v in ecol.items() if v})\n\n    uniq_keys = np.zeros(0, np.int64)\n    uniq_w = np.zeros(0, np.float64)\n    pend_k: List[np.ndarray] = []\n    pend_w: List[np.ndarray] = []\n    pending = kept_rows = 0\n    for batch in _iter_edge_batches(files.edges, [c for c in ecol.values() if c]):\n        raw_pre = np.asarray(batch[ecol["pre"]], dtype=np.int64)\n        raw_post = np.asarray(batch[ecol["post"]], dtype=np.int64)\n        w = (np.asarray(batch[ecol["weight"]], dtype=np.float64) if ecol["weight"]\n             else np.ones(len(raw_pre), np.float64))\n        report.edge_rows_read += len(w)\n\n        pre_i, ok_pre = remap_ids(sorted_ids, order, raw_pre)\n        post_i, ok_post = remap_ids(sorted_ids, order, raw_post)\n        report.dropped_unknown_pre += int((~ok_pre).sum())\n        report.dropped_unknown_post += int((ok_pre & ~ok_post).sum())\n        keep = ok_pre & ok_post\n        if drop_self_loops:\n            loops = keep & (pre_i == post_i)\n            report.dropped_self_loops += int(loops.sum())\n            keep &= ~loops\n\n        pend_k.append(post_i[keep].astype(np.int64) * n + pre_i[keep])\n        pend_w.append(w[keep])\n        pending += int(keep.sum())\n        kept_rows += int(keep.sum())\n        if pending >= merge_every:\n            uniq_keys, uniq_w = _merge_keys([uniq_keys, *pend_k], [uniq_w, *pend_w])\n            pend_k, pend_w, pending = [], [], 0\n            if verbose:\n                print(f"  rows read {report.edge_rows_read:,} -> unique pairs {len(uniq_keys):,}",\n                      flush=True)\n    uniq_keys, uniq_w = _merge_keys([uniq_keys, *pend_k], [uniq_w, *pend_w])\n    del pend_k, pend_w\n    report.duplicate_rows_merged = kept_rows - len(uniq_keys)\n\n    strong = uniq_w >= min_weight\n    report.dropped_below_min_weight = int((~strong).sum())\n    uniq_keys, uniq_w = uniq_keys[strong], uniq_w[strong]\n    pre = (uniq_keys % n).astype(np.int32)\n    post = (uniq_keys // n).astype(np.int32)\n    count = uniq_w.astype(np.float32)\n    del uniq_keys\n    report.edges_retained = len(pre)\n    report.synapses_retained = int(uniq_w.sum())\n\n    graph = ConnectomeArrays(\n        body_id=body_id, pre=pre, post=post, count=count, nt_code=nt_code,\n        nt_confidence=nt_conf, cell_type=cell_type.astype(np.int32),\n        type_names=[str(t) for t in type_names], superclass=superclass.astype(np.int16),\n        superclass_names=[str(s) for s in superclass_names], side=side, report=report,\n    )\n    report.n_type_pairs = int(len(graph.type_pair_index()[1]))\n    report.load_seconds = report.build_seconds = time.perf_counter() - t0\n    if report.accounted_rows() != report.edge_rows_read:\n        raise AssertionError(f"Edge accounting mismatch: {report.accounted_rows()} != {report.edge_rows_read}")\n    if verbose:\n        print(report.summary())\n    return graph\n\n\n_ARRAY_FIELDS = ("body_id", "pre", "post", "count", "nt_code", "nt_confidence",\n                 "cell_type", "superclass", "side")\n\n\ndef _cache_key(files: DataFiles, min_weight: float, drop_self_loops: bool,\n               statuses: Optional[Sequence[str]]) -> str:\n    parts = [str(LOADER_VERSION), str(min_weight), str(drop_self_loops),\n             ",".join(sorted(statuses)) if statuses else "*"]\n    for p in files.all():\n        st = p.stat()\n        parts += [p.name, str(st.st_size), str(int(st.st_mtime))]\n    return hashlib.sha1("|".join(parts).encode()).hexdigest()[:16]\n\n\ndef save_cache(graph: ConnectomeArrays, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.stem + ".tmp.npz")\n    np.savez(tmp, **{k: getattr(graph, k) for k in _ARRAY_FIELDS})\n    tmp.replace(path)\n    side = {"type_names": graph.type_names, "superclass_names": graph.superclass_names,\n            "report": asdict(graph.report)}\n    path.with_suffix(".json").write_text(json.dumps(side), encoding="utf-8")\n\n\ndef load_cache(path: Path) -> ConnectomeArrays:\n    t0 = time.perf_counter()\n    side = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))\n    with np.load(path) as data:\n        arrays = {k: data[k] for k in _ARRAY_FIELDS}\n    report = LoadReport(**side["report"])\n    report.from_cache = True\n    report.load_seconds = time.perf_counter() - t0\n    return ConnectomeArrays(**arrays, type_names=side["type_names"],\n                            superclass_names=side["superclass_names"], report=report)\n\n\ndef load_connectome(data_dir: str | Path = "data/connectome", cache_dir: str | Path | None = None,\n                    min_weight: float = 1.0, drop_self_loops: bool = True,\n                    statuses: Optional[Sequence[str]] = DEFAULT_STATUSES, use_cache: bool = True,\n                    verbose: bool = True) -> ConnectomeArrays:\n    """Load the connectome from ``data_dir``, building and caching it on first use.\n\n    The cache is keyed on the source files\' names, sizes and mtimes plus the\n    loader options, so a new data release or option change rebuilds it.\n    """\n    data_dir = Path(data_dir)\n    files = find_data_files(data_dir)\n    cache_dir = Path(cache_dir) if cache_dir is not None else data_dir / "cache"\n    cache = cache_dir / f"connectome_{_cache_key(files, min_weight, drop_self_loops, statuses)}.npz"\n    if use_cache and cache.exists() and cache.with_suffix(".json").exists():\n        graph = load_cache(cache)\n        if verbose:\n            print(f"Loaded cached connectome {cache.name}\\n{graph.report.summary()}")\n        return graph\n    graph = build_connectome(files, min_weight, drop_self_loops, statuses, verbose)\n    if use_cache:\n        save_cache(graph, cache)\n    return graph\n', 'meta_brain/sparse_ops.py': '"""Memory-lean sparse connectome propagation with a custom autograd (milestone M1).\n\n``y = A(values) @ x`` where ``A`` is the ``N x N`` connectome with rows =\npostsynaptic and columns = presynaptic neuron, and ``values`` holds one weight\nper edge.\n\nPlain autograd through a gather/scatter implementation saves an ``E``-sized\ntensor (``x[pre] * w``) at every call. Unrolled over ``K`` micro-steps x ``T``\nframes that is what makes BPTT through the full brain impossible. This\noperator saves only ``x`` (``N x B``) per call and recomputes edge products in\nthe backward pass, in bounded chunks:\n\n* ``grad_x      = A^T @ grad_y``                      (sparse CSR matmul)\n* ``grad_values[e] = sum_b grad_y[post_e, b] * x[pre_e, b]``   (SDDMM via\n  ``torch.sparse.sampled_addmm``; chunked gathers as a fallback)\n\n``values`` is usually derived from a much smaller parameter vector (per cell\ntype, or per connected type pair) with ordinary autograd, e.g.\n``values = base * theta[edge_pair]``; that gather is computed once per rollout\nchunk and shared across steps, so its cost is not per step.\n\nLayout: the operator works on ``(N, B)`` tensors (neurons first) so CSR\nmatmuls and row gathers need no transposes. :func:`propagate` accepts the\nmore common ``(B, N)`` layout.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\nimport warnings\n\nimport numpy as np\nimport torch\nfrom torch.autograd.function import once_differentiable\n\nwarnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta state")\n\nDEFAULT_EDGE_CHUNK = 1 << 22  # ~4M edges per chunk in the values gradient\n\n\n@dataclass\nclass SparseStructure:\n    """Fixed sparsity pattern of the connectome on one device.\n\n    Edges must be sorted by ``(post, pre)``, which is the order\n    :class:`meta_brain.graph.ConnectomeArrays` stores them in, so the edge\n    order *is* CSR order and ``values`` needs no permutation in the forward.\n    """\n\n    n: int\n    pre: torch.Tensor  # (E,) long, column index in CSR order\n    post: torch.Tensor  # (E,) long, row index in CSR order\n    crow: torch.Tensor  # (N+1,) row pointer of A\n    t_crow: torch.Tensor  # (N+1,) row pointer of A^T\n    t_col: torch.Tensor  # (E,) column index (= post) of A^T\n    t_perm: torch.Tensor  # (E,) edge index in A^T order\n    edge_chunk: int = DEFAULT_EDGE_CHUNK\n    edge_grad_kernel: str = "auto"  # "auto" | "sddmm" | "gather"\n\n    @property\n    def e(self) -> int:\n        return int(self.pre.numel())\n\n    @property\n    def device(self) -> torch.device:\n        return self.pre.device\n\n    @classmethod\n    def from_edges(cls, pre, post, n: int, device: Optional[torch.device | str] = None,\n                   index_dtype: torch.dtype = torch.int64,\n                   edge_chunk: int = DEFAULT_EDGE_CHUNK,\n                   edge_grad_kernel: str = "auto") -> "SparseStructure":\n        pre = np.asarray(pre, dtype=np.int64)\n        post = np.asarray(post, dtype=np.int64)\n        if len(pre) > 1:\n            key = post * n + pre\n            if np.any(np.diff(key) <= 0):\n                raise ValueError("Edges must be unique and sorted by (post, pre); use graph.sort_edges.")\n        if len(pre) and (pre.min() < 0 or post.min() < 0 or max(pre.max(), post.max()) >= n):\n            raise ValueError("Edge index out of range.")\n\n        crow = np.zeros(n + 1, dtype=np.int64)\n        np.cumsum(np.bincount(post, minlength=n), out=crow[1:])\n        t_perm = np.lexsort((post, pre))\n        t_crow = np.zeros(n + 1, dtype=np.int64)\n        np.cumsum(np.bincount(pre, minlength=n), out=t_crow[1:])\n\n        def to(a: np.ndarray, dtype: torch.dtype = index_dtype) -> torch.Tensor:\n            return torch.as_tensor(a).to(device=device, dtype=dtype)\n\n        return cls(n=n, pre=to(pre, torch.long), post=to(post, torch.long), crow=to(crow),\n                   t_crow=to(t_crow), t_col=to(post[t_perm]), t_perm=to(t_perm, torch.long),\n                   edge_chunk=edge_chunk, edge_grad_kernel=edge_grad_kernel)\n\n    def to(self, device: torch.device | str) -> "SparseStructure":\n        return SparseStructure(self.n, *(getattr(self, f).to(device) for f in\n                                         ("pre", "post", "crow", "t_crow", "t_col", "t_perm")),\n                               edge_chunk=self.edge_chunk, edge_grad_kernel=self.edge_grad_kernel)\n\n    def matrix(self, values: torch.Tensor) -> torch.Tensor:\n        col = self.pre.to(self.crow.dtype)\n        return torch.sparse_csr_tensor(self.crow, col, values, (self.n, self.n))\n\n    def transpose_matrix(self, values: torch.Tensor) -> torch.Tensor:\n        return torch.sparse_csr_tensor(self.t_crow, self.t_col, values[self.t_perm], (self.n, self.n))\n\n    def nbytes(self) -> int:\n        return sum(t.numel() * t.element_size() for t in\n                   (self.pre, self.post, self.crow, self.t_crow, self.t_col, self.t_perm))\n\n\ndef _edge_grad(structure: SparseStructure, grad_y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:\n    """``sum_b grad_y[post_e, b] * x[pre_e, b]`` for every edge, in CSR (= edge) order.\n\n    This is a sampled dense-dense matmul (``grad_y @ x.T`` read only at the\n    connectome\'s nonzeros), which cuSPARSE computes far faster than gathering\n    two ``E x B`` tensors.\n    """\n    kernel = structure.edge_grad_kernel\n    if kernel not in ("auto", "sddmm", "gather"):\n        raise ValueError(f"unknown edge_grad_kernel {kernel!r}")\n    if kernel != "gather":\n        try:\n            pattern = structure.matrix(torch.zeros(structure.e, dtype=grad_y.dtype, device=grad_y.device))\n            return torch.sparse.sampled_addmm(pattern, grad_y, x.t().contiguous(), beta=0.0).values()\n        except (RuntimeError, NotImplementedError):\n            if kernel == "sddmm":\n                raise\n    return _edge_grad_gather(structure, grad_y, x)\n\n\ndef _edge_grad_gather(structure: SparseStructure, grad_y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:\n    """Fallback: the same edge gradient from chunked row gathers."""\n    out = torch.empty(structure.e, dtype=grad_y.dtype, device=grad_y.device)\n    step = max(1, int(structure.edge_chunk))\n    for s in range(0, structure.e, step):\n        e = min(s + step, structure.e)\n        out[s:e] = (grad_y.index_select(0, structure.post[s:e]) *\n                    x.index_select(0, structure.pre[s:e])).sum(dim=1)\n    return out\n\n\nclass _SparsePropagate(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:\n        ctx.structure = structure\n        ctx.save_for_backward(x, values)\n        return torch.sparse.mm(structure.matrix(values), x)\n\n    @staticmethod\n    @once_differentiable\n    def backward(ctx, grad_y: torch.Tensor):\n        x, values = ctx.saved_tensors\n        structure: SparseStructure = ctx.structure\n        grad_y = grad_y.contiguous()\n        grad_x = grad_values = None\n        if ctx.needs_input_grad[0]:\n            grad_x = torch.sparse.mm(structure.transpose_matrix(values), grad_y)\n        if ctx.needs_input_grad[1]:\n            grad_values = _edge_grad(structure, grad_y, x)\n        return grad_x, grad_values, None\n\n\ndef sparse_propagate(x_nb: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:\n    """``A(values) @ x`` for ``x`` of shape ``(N, B)``."""\n    if x_nb.dim() != 2 or x_nb.shape[0] != structure.n:\n        raise ValueError(f"x must be (N={structure.n}, B), got {tuple(x_nb.shape)}")\n    if values.shape != (structure.e,):\n        raise ValueError(f"values must be (E={structure.e},), got {tuple(values.shape)}")\n    return _SparsePropagate.apply(x_nb.contiguous(), values, structure)\n\n\ndef propagate(x_bn: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:\n    """Batch-first convenience wrapper: ``(B, N) -> (B, N)``."""\n    return sparse_propagate(x_bn.t(), values, structure).t()\n\n\ndef reference_propagate(x_nb: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:\n    """Plain-autograd gather/scatter reference used by the tests and the benchmark."""\n    msg = x_nb.index_select(0, structure.pre) * values.unsqueeze(1)\n    out = torch.zeros_like(x_nb)\n    return out.index_add(0, structure.post, msg)\n\n\n__all__ = [\n    "SparseStructure",\n    "sparse_propagate",\n    "propagate",\n    "reference_propagate",\n]\n', 'meta_brain/mario_rl.py': '"""Super Mario Bros with a frozen connectome reservoir and a trained motor readout.\n\nShared by ``scripts/full_brain_mario_v2.py`` and the Mario V2 Colab notebook\n(which embeds this file). The scientific setup is unchanged from V2: the\nconnectome and the visual projection are frozen; only a linear readout of the\nmotor neurons is trained. What changed is everything around it that kept the\nfirst full run from learning:\n\n* **Frame skip.** One agent step repeats the action for ``frame_skip`` NES\n  frames, so a jump can be held long enough to clear a Goomba or a pipe.\n* **Episode cut-offs.** An episode ends when Mario makes no progress for\n  ``stuck_steps`` agent steps instead of pushing against a pipe until the\n  step limit.\n* **No-op starts.** A random number of idle frames at reset makes episodes\n  differ, so evaluation episodes are not five copies of one trajectory.\n* **PPO on the readout.** Batched environments, a value baseline from the\n  same motor neurons and clipped updates replace one-episode REINFORCE.\n* **Normalized motor features.** Motor activity is standardized with\n  statistics from a random-policy warm-up, so the readout sees signals of\n  usable scale.\n* **Playable videos.** Frames are encoded as H.264 (yuv420p) with ffmpeg; the\n  old ``mp4v`` files do not play in browsers.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport shutil\nimport subprocess\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nfrom .sparse_ops import SparseStructure\n\n# --------------------------------------------------------------------------- #\n# Environment\n# --------------------------------------------------------------------------- #\n\n\ndef make_mario_env(stage: str = "1-1", actions: str = "right_only"):\n    """``gym-super-mario-bros`` stage wrapped in a joypad action space."""\n    import gym_super_mario_bros\n    from gym_super_mario_bros import actions as action_sets\n    from nes_py.wrappers import JoypadSpace\n\n    table = {"right_only": action_sets.RIGHT_ONLY, "simple": action_sets.SIMPLE_MOVEMENT}[actions]\n    env = JoypadSpace(gym_super_mario_bros.make(f"SuperMarioBros-{stage}-v0"), table)\n    return env, ["+".join(a) for a in table]\n\n\ndef _reset(env, seed=None):\n    if seed is None:\n        out = env.reset()\n    else:\n        try:\n            out = env.reset(seed=seed)\n        except TypeError as exc:\n            if "unexpected keyword argument \'seed\'" not in str(exc):\n                raise\n            out = env.reset()  # compatibility with legacy Gym and simple test environments\n    return out[0] if isinstance(out, tuple) else out\n\n\ndef _step(env, action: int):\n    out = env.step(int(action))\n    if len(out) == 5:\n        obs, reward, terminated, truncated, info = out\n        return obs, float(reward), bool(terminated or truncated), dict(\n            info, terminated=bool(terminated), env_truncated=bool(truncated))\n    obs, reward, done, info = out\n    return obs, float(reward), bool(done), dict(info, terminated=bool(done), env_truncated=False)\n\n\nclass MarioRunner:\n    """One environment with frame skip, no-op starts, progress cut-off and frame recording."""\n\n    def __init__(self, env, frame_skip: int = 4, noop_max: int = 30, stuck_steps: int = 60,\n                 max_steps: int = 1500, seed: int = 0, record_stride: int = 1):\n        self.env = env\n        self.frame_skip = frame_skip\n        self.noop_max = noop_max\n        self.stuck_steps = stuck_steps\n        self.max_steps = max_steps\n        self.rng = np.random.default_rng(seed)\n        self.reset_rng = np.random.default_rng(seed + 1_000_000)\n        self.record = False\n        self.record_stride = max(1, record_stride)\n        self.capture_count = 0\n        self.frames: List[np.ndarray] = []\n        self.info: Dict = {}\n        self.steps = self.since_progress = 0\n        self.best_x = 0.0\n        self.episode_reward = 0.0\n\n    def _capture(self, obs) -> None:\n        if self.record:\n            if self.capture_count % self.record_stride == 0:\n                self.frames.append(np.asarray(obs, dtype=np.uint8).copy())\n            self.capture_count += 1\n\n    def reset(self, record: bool = False):\n        self.record = record\n        self.frames = []\n        self.capture_count = 0\n        obs = _reset(self.env, seed=int(self.reset_rng.integers(0, 2**31)))\n        self._capture(obs)\n        info: Dict = {}\n        for _ in range(int(self.rng.integers(0, self.noop_max + 1))):\n            obs, _, done, info = _step(self.env, 0)\n            self._capture(obs)\n            if done:\n                obs, info = _reset(self.env), {}\n        self.steps = self.since_progress = 0\n        self.best_x = float(info.get("x_pos", 0))\n        self.episode_reward = 0.0\n        self.info = info\n        return obs\n\n    def step(self, action: int):\n        total, done, info = 0.0, False, {}\n        for _ in range(self.frame_skip):\n            obs, reward, done, info = _step(self.env, action)\n            total += reward\n            self._capture(obs)\n            if done:\n                break\n        self.steps += 1\n        self.episode_reward += total\n        x = float(info.get("x_pos", self.best_x))\n        if x > self.best_x + 1:\n            self.best_x, self.since_progress = x, 0\n        else:\n            self.since_progress += 1\n        flag = bool(info.get("flag_get", False))\n        truncated = bool(info.get("env_truncated", False)) or (\n            not done and (self.since_progress >= self.stuck_steps or self.steps >= self.max_steps))\n        terminated = bool(info.get("terminated", done))\n        info = dict(info, flag_get=flag, died=bool(terminated and not flag),\n                    terminated=terminated, truncated=truncated)\n        self.info = info\n        return obs, total, done or truncated, info\n\n    def episode_summary(self) -> Dict:\n        i = self.info\n        return {"reward": round(self.episode_reward, 2), "steps": self.steps,\n                "x_pos": float(i.get("x_pos", self.best_x)), "max_x": self.best_x,\n                "flag_get": bool(i.get("flag_get", False)), "died": bool(i.get("died", False)),\n                "truncated": bool(i.get("truncated", False))}\n\n\n# --------------------------------------------------------------------------- #\n# Vision and the connectome reservoir\n# --------------------------------------------------------------------------- #\n\n\ndef encode_frame(frame, previous: Optional[np.ndarray] = None, size: Tuple[int, int] = (32, 32),\n                 crop_top: int = 32) -> Tuple[np.ndarray, np.ndarray]:\n    """Grayscale playfield (HUD cropped), downsampled, plus the change since the last frame."""\n    from PIL import Image\n\n    a = np.asarray(frame)\n    if a.ndim == 3:\n        a = a[..., :3].mean(axis=-1)\n    a = a[crop_top:]\n    small = np.asarray(Image.fromarray(a.astype(np.uint8)).resize((size[1], size[0]), Image.BILINEAR),\n                       dtype=np.float32) / 127.5 - 1.0\n    delta = small if previous is None else small - previous\n    return np.concatenate([small.ravel(), delta.ravel()]).astype(np.float32), small\n\n\ndef code_dim(size: Tuple[int, int] = (32, 32)) -> int:\n    return 2 * size[0] * size[1]\n\n\nVISION_MODES = ("real", "zero", "clock")\n\n\nclass VisualInput:\n    """What the sensory neurons receive; the blind modes are controls for "does the readout use vision?".\n\n    * ``real``: the game frame (:func:`encode_frame`).\n    * ``zero``: nothing. With no drive the reservoir stays silent, so the\n      readout can only learn a fixed action distribution.\n    * ``clock``: a fixed random code for each agent step since reset, the same\n      in every episode and unrelated to the screen. The reservoir is as active\n      as with real input and carries elapsed time, so the readout can learn an\n      open-loop rhythm of jumps but cannot react to what is on screen.\n    """\n\n    def __init__(self, mode: str = "real", size: Tuple[int, int] = (32, 32), seed: int = 0):\n        if mode not in VISION_MODES:\n            raise ValueError(f"vision must be one of {VISION_MODES}, got {mode!r}")\n        self.mode, self.size, self.seed = mode, size, seed\n        self.dim = code_dim(size)\n        self._clock: Dict[int, np.ndarray] = {}\n\n    def __call__(self, obs, previous: Optional[np.ndarray], step: int) -> Tuple[np.ndarray, Optional[np.ndarray]]:\n        if self.mode == "real":\n            return encode_frame(obs, previous, self.size)\n        if self.mode == "zero":\n            return np.zeros(self.dim, np.float32), None\n        code = self._clock.get(step)\n        if code is None:\n            code = np.random.default_rng([self.seed, step]).uniform(-1, 1, self.dim).astype(np.float32)\n            self._clock[step] = code\n        return code, None\n\n\nclass ConnectomeReservoir:\n    """Frozen recurrent dynamics on the connectome; returns motor-neuron activity.\n\n    ``state <- decay * state + (1 - decay) * tanh(gain * A @ state + drive)``,\n    repeated ``micro_steps`` times per agent step, with ``drive`` a fixed\n    random projection of the visual code onto the sensory neurons.\n\n    ``gain`` defaults to 0.5. The layer probe sweep showed that at the V2\n    value of 1.6 the network saturates (up to 45% of central-brain units at\n    |state| > 0.9) and the screen is no longer decodable beyond the first\n    optic-lobe synapse; at 0.5 it still reaches the descending neurons.\n    """\n\n    def __init__(self, structure: SparseStructure, weights: torch.Tensor, sensory: Sequence[int],\n                 motor: Sequence[int], input_dim: int, micro_steps: int = 2, decay: float = 0.9,\n                 gain: float = 0.5, input_gain: float = 1.0, seed: int = 0):\n        device = structure.device\n        self.n = structure.n\n        self.device = device\n        self.A = structure.matrix(weights.to(device=device, dtype=torch.float32))\n        self.sensory = torch.as_tensor(np.asarray(sensory), dtype=torch.long, device=device)\n        self.motor = torch.as_tensor(np.asarray(motor), dtype=torch.long, device=device)\n        gen = torch.Generator(device="cpu").manual_seed(seed)\n        self.proj = (torch.randn(input_dim, len(self.sensory), generator=gen) / math.sqrt(input_dim)).to(device)\n        self.micro_steps, self.decay, self.gain, self.input_gain = micro_steps, decay, gain, input_gain\n\n    @property\n    def n_motor(self) -> int:\n        return int(self.motor.numel())\n\n    def initial_state(self, batch: int) -> torch.Tensor:\n        return torch.zeros(self.n, batch, device=self.device)\n\n    @torch.no_grad()\n    def step(self, codes: torch.Tensor, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        """``codes`` is ``(B, input_dim)``, ``state`` is ``(N, B)``; returns ``(B, n_motor)`` features."""\n        drive = torch.zeros_like(state)\n        drive.index_add_(0, self.sensory, (codes @ self.proj).t() * self.input_gain)\n        for _ in range(self.micro_steps):\n            msg = torch.sparse.mm(self.A, state)\n            state = self.decay * state + (1.0 - self.decay) * torch.tanh(self.gain * msg + drive)\n        return state.index_select(0, self.motor).t().contiguous(), state\n\n\n# --------------------------------------------------------------------------- #\n# Readout policy and PPO\n# --------------------------------------------------------------------------- #\n\n\nclass ReadoutPolicy(nn.Module):\n    """Linear policy and value heads on standardized motor-neuron activity."""\n\n    def __init__(self, n_features: int, n_actions: int, seed: int = 0):\n        super().__init__()\n        gen = torch.Generator().manual_seed(seed)\n        self.register_buffer("mean", torch.zeros(n_features))\n        self.register_buffer("std", torch.ones(n_features))\n        self.pi = nn.Linear(n_features, n_actions)\n        self.v = nn.Linear(n_features, 1)\n        with torch.no_grad():\n            self.pi.weight.copy_(torch.randn(n_actions, n_features, generator=gen) * 0.01)\n            self.pi.bias.zero_()\n            self.v.weight.zero_()\n            self.v.bias.zero_()\n\n    def fit_normalization(self, features: torch.Tensor) -> Dict[str, float]:\n        """Standardize with warm-up statistics; near-constant neurons are scaled, not amplified."""\n        std = features.std(dim=0)\n        floor = 1e-2 * float(std.mean()) + 1e-6\n        self.mean.copy_(features.mean(dim=0))\n        self.std.copy_(std.clamp_min(floor))\n        return {"motor_features": int(features.shape[1]),\n                "responsive_fraction": round(float((std > floor).float().mean()), 4),\n                "mean_feature_std": float(std.mean()), "mean_abs_activity": float(features.abs().mean())}\n\n    def forward(self, features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        z = (features - self.mean) / self.std\n        return self.pi(z), self.v(z).squeeze(-1)\n\n\n@dataclass\nclass PPOConfig:\n    updates: int = 300\n    rollout: int = 128\n    gamma: float = 0.99\n    lam: float = 0.95\n    clip: float = 0.2\n    epochs: int = 4\n    minibatches: int = 4\n    lr: float = 3e-4\n    entropy: float = 0.01\n    value_coef: float = 0.5\n    max_grad_norm: float = 0.5\n    reward_scale: float = 0.1\n    warmup_steps: int = 64\n\n\nclass BatchedRollout:\n    """Steps ``B`` runners through the reservoir in lockstep, resetting finished episodes."""\n\n    def __init__(self, runners: Sequence[MarioRunner], reservoir: ConnectomeReservoir,\n                 size: Tuple[int, int] = (32, 32), vision: Optional[VisualInput] = None):\n        self.runners, self.reservoir, self.size = list(runners), reservoir, size\n        self.vision = vision or VisualInput("real", size)\n        self.prev: List[Optional[np.ndarray]] = [None] * len(self.runners)\n        self.state = reservoir.initial_state(len(self.runners))\n        self.episodes: List[Dict] = []\n        codes = [self._encode(i, r.reset()) for i, r in enumerate(self.runners)]\n        self.features = self._advance(codes)\n\n    def _encode(self, i: int, obs) -> np.ndarray:\n        code, self.prev[i] = self.vision(obs, self.prev[i], self.runners[i].steps)\n        return code\n\n    def _advance(self, codes: List[np.ndarray]) -> torch.Tensor:\n        batch = torch.as_tensor(np.stack(codes), device=self.reservoir.device)\n        features, self.state = self.reservoir.step(batch, self.state)\n        return features\n\n    def step(self, actions: Sequence[int]) -> Tuple[np.ndarray, np.ndarray]:\n        rewards = np.zeros(len(self.runners), np.float32)\n        dones = np.zeros(len(self.runners), np.float32)\n        codes = []\n        for i, (runner, a) in enumerate(zip(self.runners, actions)):\n            obs, reward, done, _ = runner.step(int(a))\n            rewards[i], dones[i] = reward, float(done)\n            if done:\n                self.episodes.append(runner.episode_summary())\n                obs = runner.reset()\n                self.prev[i] = None\n                self.state[:, i] = 0.0\n            codes.append(self._encode(i, obs))\n        self.features = self._advance(codes)\n        return rewards, dones\n\n\n@torch.no_grad()\ndef warmup_normalization(rollout: BatchedRollout, policy: ReadoutPolicy, steps: int, n_actions: int,\n                         seed: int = 0) -> Dict[str, float]:\n    """Fit the feature normalization on a random-action rollout."""\n    rng = np.random.default_rng(seed)\n    feats = [rollout.features]\n    for _ in range(steps):\n        rollout.step(rng.integers(0, n_actions, len(rollout.runners)))\n        feats.append(rollout.features)\n    rollout.episodes.clear()\n    return policy.fit_normalization(torch.cat(feats).float())\n\n\ndef train_ppo(rollout: BatchedRollout, policy: ReadoutPolicy, cfg: PPOConfig,\n              log: Optional[Callable[[str], None]] = print, log_every: int = 10) -> List[Dict]:\n    """PPO on the readout heads; the reservoir is frozen, so stored features are reused for every epoch."""\n    device = rollout.reservoir.device\n    params = list(policy.pi.parameters()) + list(policy.v.parameters())\n    opt = torch.optim.Adam(params, lr=cfg.lr)\n    n_envs = len(rollout.runners)\n    history: List[Dict] = []\n    for update in range(cfg.updates):\n        feats, acts, logps, vals = [], [], [], []\n        rews = np.zeros((cfg.rollout, n_envs), np.float32)\n        dones = np.zeros((cfg.rollout, n_envs), np.float32)\n        first_episode = len(rollout.episodes)\n        for t in range(cfg.rollout):\n            f = rollout.features\n            with torch.no_grad():\n                logits, value = policy(f)\n                dist = torch.distributions.Categorical(logits=logits)\n                a = dist.sample()\n            feats.append(f)\n            acts.append(a)\n            logps.append(dist.log_prob(a))\n            vals.append(value)\n            rews[t], dones[t] = rollout.step(a.tolist())\n        with torch.no_grad():\n            last_value = policy(rollout.features)[1].cpu().numpy()\n\n        values = torch.stack(vals).cpu().numpy()\n        adv = np.zeros_like(rews)\n        gae = np.zeros(n_envs, np.float32)\n        for t in reversed(range(cfg.rollout)):\n            next_value = last_value if t == cfg.rollout - 1 else values[t + 1]\n            nonterminal = 1.0 - dones[t]\n            delta = rews[t] * cfg.reward_scale + cfg.gamma * next_value * nonterminal - values[t]\n            gae = delta + cfg.gamma * cfg.lam * nonterminal * gae\n            adv[t] = gae\n        returns = adv + values\n\n        F_ = torch.cat(feats)\n        A_ = torch.cat(acts)\n        LP_ = torch.cat(logps)\n        ADV = torch.as_tensor(adv.reshape(-1), device=device)\n        RET = torch.as_tensor(returns.reshape(-1), device=device)\n        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)\n        n = len(A_)\n        mb = max(1, n // cfg.minibatches)\n        stats = {"policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0, "clip_frac": 0.0}\n        count = 0\n        for _ in range(cfg.epochs):\n            perm = torch.randperm(n, device=device)\n            for s in range(0, n, mb):\n                idx = perm[s:s + mb]\n                logits, value = policy(F_[idx])\n                dist = torch.distributions.Categorical(logits=logits)\n                ratio = torch.exp(dist.log_prob(A_[idx]) - LP_[idx])\n                pg = -torch.min(ratio * ADV[idx], ratio.clamp(1 - cfg.clip, 1 + cfg.clip) * ADV[idx]).mean()\n                vloss = (value - RET[idx]).pow(2).mean()\n                ent = dist.entropy().mean()\n                loss = pg + cfg.value_coef * vloss - cfg.entropy * ent\n                opt.zero_grad()\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)\n                opt.step()\n                stats["policy_loss"] += float(pg.detach())\n                stats["value_loss"] += float(vloss.detach())\n                stats["entropy"] += float(ent.detach())\n                stats["clip_frac"] += float(((ratio - 1).abs() > cfg.clip).float().mean())\n                count += 1\n        finished = rollout.episodes[first_episode:]\n        row = {"update": update, "agent_steps": (update + 1) * cfg.rollout * n_envs,\n               "episodes": len(finished),\n               "mean_reward": float(np.mean([e["reward"] for e in finished])) if finished else float("nan"),\n               "mean_max_x": float(np.mean([e["max_x"] for e in finished])) if finished else float("nan"),\n               "best_max_x": float(max([e["max_x"] for e in finished], default=float("nan"))),\n               "flags": int(sum(e["flag_get"] for e in finished)),\n               **{k: v / max(count, 1) for k, v in stats.items()}}\n        history.append(row)\n        if log and (update == 0 or (update + 1) % log_every == 0):\n            log(f"update {update + 1:4d}/{cfg.updates} steps={row[\'agent_steps\']:,} "\n                f"episodes={row[\'episodes\']:3d} mean_x={row[\'mean_max_x\']:.0f} best_x={row[\'best_max_x\']:.0f} "\n                f"flags={row[\'flags\']} entropy={row[\'entropy\']:.2f}")\n    return history\n\n\n# --------------------------------------------------------------------------- #\n# Evaluation and video\n# --------------------------------------------------------------------------- #\n\n\ndef episode_key(row: Dict) -> Tuple:\n    """Best episode: a clean flag first, then the furthest progress, then reward."""\n    return (int(row["flag_get"] and not row["died"]), row["max_x"], row["reward"])\n\n\n@torch.no_grad()\ndef evaluate(runner: MarioRunner, reservoir: ConnectomeReservoir, policy: ReadoutPolicy,\n             episodes: int, greedy: bool = True, record: bool = True,\n             size: Tuple[int, int] = (32, 32), seed: int = 0,\n             vision: Optional[VisualInput] = None) -> Tuple[List[Dict], List[np.ndarray]]:\n    """Run ``episodes`` episodes; returns per-episode rows and the frames of the best one."""\n    vision = vision or VisualInput("real", size)\n    gen = torch.Generator(device="cpu").manual_seed(seed)\n    rows, best_frames, best = [], [], None\n    for _ in range(episodes):\n        obs = runner.reset(record=record)\n        prev, state, done = None, reservoir.initial_state(1), False\n        while not done:\n            code, prev = vision(obs, prev, runner.steps)\n            features, state = reservoir.step(torch.as_tensor(code[None], device=reservoir.device), state)\n            logits, _ = policy(features)\n            if greedy:\n                action = int(logits.argmax(-1))\n            else:\n                action = int(torch.multinomial(torch.softmax(logits.cpu(), -1), 1, generator=gen))\n            obs, _, done, _ = runner.step(action)\n        row = runner.episode_summary()\n        rows.append(row)\n        if record and (best is None or episode_key(row) > episode_key(best)):\n            best, best_frames = row, runner.frames\n    return rows, best_frames\n\n\ndef summarize(rows: List[Dict]) -> Dict:\n    return {"episodes": rows,\n            "reward_mean": float(np.mean([r["reward"] for r in rows])),\n            "max_x_mean": float(np.mean([r["max_x"] for r in rows])),\n            "max_x_best": float(max(r["max_x"] for r in rows)),\n            "success_rate": float(np.mean([r["flag_get"] and not r["died"] for r in rows])),\n            "death_rate": float(np.mean([r["died"] for r in rows])),\n            "all_success_no_death": bool(all(r["flag_get"] and not r["died"] for r in rows))}\n\n\ndef save_video(frames: Sequence[np.ndarray], path: str | Path, fps: int = 60, scale: int = 2) -> str:\n    """Write an H.264/yuv420p MP4 that browsers and Colab can play; returns the codec used.\n\n    Falls back to OpenCV\'s ``mp4v`` (which most browsers cannot play) only when\n    no ``ffmpeg`` binary is available.\n    """\n    if not frames:\n        raise ValueError("no frames to write")\n    path = Path(path)\n    first = np.asarray(frames[0])\n    h, w = first.shape[:2]\n    ffmpeg = shutil.which("ffmpeg")\n    if ffmpeg:\n        cmd = [ffmpeg, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",\n               "-s", f"{w}x{h}", "-r", str(fps), "-i", "-",\n               "-vf", f"scale={w * scale}:{h * scale}:flags=neighbor",\n               "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", "-movflags", "+faststart", str(path)]\n        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)\n        try:\n            for frame in frames:\n                a = np.asarray(frame, dtype=np.uint8)\n                if a.ndim == 2:\n                    a = np.repeat(a[..., None], 3, axis=-1)\n                proc.stdin.write(np.ascontiguousarray(a[..., :3]).tobytes())\n        finally:\n            proc.stdin.close()\n            err = proc.stderr.read().decode(errors="replace")\n            proc.wait()\n        if proc.returncode != 0:\n            raise RuntimeError(f"ffmpeg failed: {err.strip()[:300]}")\n        return "h264"\n    import cv2\n\n    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w * scale, h * scale))\n    try:\n        for frame in frames:\n            a = np.asarray(frame, dtype=np.uint8)\n            if a.ndim == 2:\n                a = np.repeat(a[..., None], 3, axis=-1)\n            a = cv2.resize(a[..., :3], (w * scale, h * scale), interpolation=cv2.INTER_NEAREST)\n            writer.write(cv2.cvtColor(a, cv2.COLOR_RGB2BGR))\n    finally:\n        writer.release()\n    return "mp4v"\n\n\ndef config_dict(cfg: PPOConfig) -> Dict:\n    return asdict(cfg)\n\n\n# --------------------------------------------------------------------------- #\n# Whole experiment\n# --------------------------------------------------------------------------- #\n\n\n@dataclass\nclass MarioSettings:\n    n_envs: int = 8\n    frame_skip: int = 4\n    micro_steps: int = 2\n    size: Tuple[int, int] = (32, 32)\n    noop_max: int = 30\n    stuck_steps: int = 60\n    max_steps: int = 1500\n    eval_episodes: int = 10\n    video_fps: int = 60\n    seed: int = 123\n    vision: str = "real"  # "real" | "zero" | "clock", see VisualInput\n    gain: float = 0.5  # recurrent gain of the reservoir (V2 used 1.6, which saturates; see ConnectomeReservoir)\n    input_gain: float = 1.0\n\n\ndef _plot_history(history: List[Dict], path: Path) -> None:\n    import matplotlib.pyplot as plt\n\n    steps = [h["agent_steps"] for h in history]\n    fig, axes = plt.subplots(1, 3, figsize=(16, 4))\n    mean_x = np.array([h["mean_max_x"] for h in history], dtype=float)\n    best_x = np.array([h["best_max_x"] for h in history], dtype=float)\n    axes[0].plot(steps, mean_x, alpha=0.4, label="mean max x per update")\n    if len(mean_x) >= 5 and np.isfinite(mean_x).any():\n        filled = np.where(np.isfinite(mean_x), mean_x, np.nanmean(mean_x))\n        axes[0].plot(steps[4:], np.convolve(filled, np.ones(5) / 5, mode="valid"), label="5-update average")\n    axes[0].plot(steps, best_x, ".", ms=3, alpha=0.5, label="best episode")\n    axes[0].set_title("Progress in World 1-1 (x position)")\n    axes[0].legend(fontsize=8)\n    axes[1].plot(steps, [h["mean_reward"] for h in history])\n    axes[1].set_title("Mean episode reward")\n    axes[2].plot(steps, [h["entropy"] for h in history], label="policy entropy")\n    axes[2].plot(steps, [h["clip_frac"] for h in history], label="PPO clip fraction")\n    axes[2].set_title("Readout optimization")\n    axes[2].legend(fontsize=8)\n    for ax in axes:\n        ax.set_xlabel("agent steps")\n        ax.grid(alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\n\ndef run_experiment(structure: SparseStructure, weights: torch.Tensor, sensory: Sequence[int],\n                   motor: Sequence[int], env_factory: Callable[[], object], out_dir: str | Path,\n                   settings: Optional[MarioSettings] = None, ppo: Optional[PPOConfig] = None,\n                   graph_info: Optional[Dict] = None, n_actions: int = 5,\n                   log: Optional[Callable[[str], None]] = print, prefix: str = "full_brain_mario_v2") -> Dict:\n    """Pre-training evaluation, PPO on the readout, post-training evaluation and all artifacts."""\n    import json\n    import time\n\n    import pandas as pd\n\n    settings = settings or MarioSettings()\n    ppo = ppo or PPOConfig()\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(settings.seed)\n    started = time.time()\n    reservoir = ConnectomeReservoir(structure, weights, sensory, motor, code_dim(settings.size),\n                                    micro_steps=settings.micro_steps, gain=settings.gain,\n                                    input_gain=settings.input_gain, seed=settings.seed)\n\n    def runner(i: int) -> MarioRunner:\n        return MarioRunner(env_factory(), settings.frame_skip, settings.noop_max, settings.stuck_steps,\n                           settings.max_steps, seed=settings.seed + i)\n\n    vision = VisualInput(settings.vision, settings.size, settings.seed)\n    rollout = BatchedRollout([runner(i) for i in range(settings.n_envs)], reservoir, settings.size, vision)\n    policy = ReadoutPolicy(reservoir.n_motor, n_actions, seed=settings.seed).to(reservoir.device)\n    diagnostics = warmup_normalization(rollout, policy, ppo.warmup_steps, n_actions, seed=settings.seed)\n    if log:\n        log(f"motor readout: {diagnostics}")\n\n    eval_runner = runner(10_000)\n    pre_rows, pre_frames = evaluate(eval_runner, reservoir, policy, settings.eval_episodes,\n                                    size=settings.size, seed=settings.seed, vision=vision)\n    pre = summarize(pre_rows)\n    if log:\n        log(f"pre-training: mean max x={pre[\'max_x_mean\']:.0f} best={pre[\'max_x_best\']:.0f}")\n\n    t_train = time.time()\n    history = train_ppo(rollout, policy, ppo, log=log)\n    train_minutes = (time.time() - t_train) / 60\n\n    post_rows, post_frames = evaluate(eval_runner, reservoir, policy, settings.eval_episodes,\n                                      size=settings.size, seed=settings.seed + 1, vision=vision)\n    post = summarize(post_rows)\n    # Greedy evaluation turns a blind policy into one fixed action, so every condition is\n    # also evaluated with sampled actions; the comparison between conditions uses both.\n    sampled_rows, sampled_frames = evaluate(eval_runner, reservoir, policy, settings.eval_episodes,\n                                            greedy=False, size=settings.size, seed=settings.seed + 2,\n                                            vision=vision)\n    post_sampled = summarize(sampled_rows)\n    if log:\n        log(f"post-training [{settings.vision}]: greedy mean max x={post[\'max_x_mean\']:.0f} "\n            f"best={post[\'max_x_best\']:.0f} | sampled mean max x={post_sampled[\'max_x_mean\']:.0f} "\n            f"best={post_sampled[\'max_x_best\']:.0f} | flags={sum(r[\'flag_get\'] for r in post_rows)}"\n            f"/{len(post_rows)}")\n\n    codec = save_video(pre_frames, out / f"{prefix}_pre.mp4", settings.video_fps)\n    save_video(post_frames, out / f"{prefix}_post.mp4", settings.video_fps)\n    # The trained policy is stochastic; its best behavior only shows up with sampled actions.\n    save_video(sampled_frames, out / f"{prefix}_post_sampled.mp4", settings.video_fps)\n    _plot_history(history, out / f"{prefix}_curves.png")\n    pd.DataFrame(history).to_csv(out / f"{prefix}_training.csv", index=False)\n    pd.DataFrame(rollout.episodes).to_csv(out / f"{prefix}_episodes.csv", index=False)\n    torch.save({"policy": {k: v.detach().cpu() for k, v in policy.state_dict().items()},\n                "settings": asdict(settings), "ppo": asdict(ppo)}, out / f"{prefix}_readout.pt")\n    steps = history[-1]["agent_steps"] if history else 0\n    results = {\n        "experiment": "Meta-Brain full-connectome Mario V2 (PPO readout)",\n        "graph": graph_info or {},\n        "trainable": "linear readout of motor neurons (policy + value); connectome and visual projection frozen",\n        "vision": settings.vision,\n        "observation": {"real": f"{settings.size[0]}x{settings.size[1]} grayscale playfield + frame difference",\n                        "zero": "none (blind control: zero sensory drive)",\n                        "clock": "blind control: fixed random code per step since reset"}[settings.vision],\n        "settings": asdict(settings),\n        "ppo": asdict(ppo),\n        "motor_readout_diagnostics": diagnostics,\n        "training": {"agent_steps": steps, "nes_frames": steps * settings.frame_skip,\n                     "episodes": len(rollout.episodes), "minutes": round(train_minutes, 2),\n                     "agent_steps_per_second": round(steps / max(train_minutes * 60, 1e-9), 1)},\n        "pre_eval": pre,\n        "post_eval": post,\n        "post_eval_sampled": post_sampled,\n        "history": history,\n        "video_codec": codec,\n        "videos": {"pre": f"{prefix}_pre.mp4 (best greedy episode before training)",\n                   "post": f"{prefix}_post.mp4 (best greedy episode after training)",\n                   "post_sampled": f"{prefix}_post_sampled.mp4 (best sampled-action episode after training)"},\n        "runtime_minutes": round((time.time() - started) / 60, 2),\n        "claim_status": ("DEMONSTRATED_SUCCESS" if post["all_success_no_death"]\n                         else "RUN_COMPLETED_BUT_SUCCESS_NOT_DEMONSTRATED"),\n    }\n    (out / f"{prefix}_results.json").write_text(json.dumps(results, indent=2, default=float), encoding="utf-8")\n    return results\n\n\ndef compare_vision_conditions(results: Dict[str, Dict], out_dir: str | Path,\n                              name: str = "mario_vision_controls") -> Dict:\n    """Table, plot and JSON comparing runs that differ only in ``settings.vision``.\n\n    Reads ``post_eval`` (greedy), ``post_eval_sampled`` and ``history`` from\n    each :func:`run_experiment` result. This is one seed per condition, so it\n    can show a large effect but cannot settle a small one.\n    """\n    import json\n\n    import matplotlib.pyplot as plt\n\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for cond, r in results.items():\n        hist = r.get("history", [])\n        tail = [h["mean_max_x"] for h in hist[-max(1, len(hist) // 10):] if np.isfinite(h["mean_max_x"])]\n        rows.append({\n            "condition": cond,\n            "pre_greedy_mean_x": r["pre_eval"]["max_x_mean"],\n            "post_greedy_mean_x": r["post_eval"]["max_x_mean"],\n            "post_greedy_best_x": r["post_eval"]["max_x_best"],\n            "post_sampled_mean_x": r["post_eval_sampled"]["max_x_mean"],\n            "post_sampled_best_x": r["post_eval_sampled"]["max_x_best"],\n            "post_death_rate": r["post_eval"]["death_rate"],\n            "train_final_mean_x": float(np.mean(tail)) if tail else float("nan"),\n            "flags": int(sum(e["flag_get"] for e in r["post_eval"]["episodes"])),\n            "responsive_motor_fraction": r["motor_readout_diagnostics"]["responsive_fraction"],\n        })\n\n    def best_of(row):\n        return max(row["post_greedy_mean_x"], row["post_sampled_mean_x"])\n\n    summary = {"conditions": rows, "single_seed": True}\n    real = next((r for r in rows if r["condition"] == "real"), None)\n    if real is not None:\n        for row in rows:\n            if row["condition"] != "real":\n                summary[f"real_minus_{row[\'condition\']}_mean_x"] = round(best_of(real) - best_of(row), 1)\n                summary[f"real_minus_{row[\'condition\']}_train_x"] = round(\n                    real["train_final_mean_x"] - row["train_final_mean_x"], 1)\n\n    colors = {"real": "#4878d0", "zero": "#d65f5f", "clock": "#ee854a"}\n    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))\n    for cond, r in results.items():\n        hist = r.get("history", [])\n        if not hist:\n            continue\n        steps = np.array([h["agent_steps"] for h in hist])\n        x = np.array([h["mean_max_x"] for h in hist], dtype=float)\n        x = np.where(np.isfinite(x), x, np.nanmean(x))\n        k = min(10, len(x))\n        axes[0].plot(steps[k - 1:], np.convolve(x, np.ones(k) / k, mode="valid"),\n                     color=colors.get(cond), label=cond)\n    axes[0].set_title("Training: mean max x per update (10-update average)")\n    axes[0].set_xlabel("agent steps")\n    axes[0].legend()\n    axes[0].grid(alpha=0.25)\n    idx = np.arange(len(rows))\n    axes[1].bar(idx - 0.2, [r["post_greedy_mean_x"] for r in rows], 0.4, label="greedy",\n                color=[colors.get(r["condition"], "gray") for r in rows])\n    axes[1].bar(idx + 0.2, [r["post_sampled_mean_x"] for r in rows], 0.4, label="sampled",\n                color=[colors.get(r["condition"], "gray") for r in rows], alpha=0.5)\n    axes[1].set_xticks(idx, [r["condition"] for r in rows])\n    axes[1].set_title("After training: mean max x over evaluation episodes")\n    axes[1].legend()\n    axes[1].grid(alpha=0.25, axis="y")\n    fig.tight_layout()\n    fig.savefig(out / f"{name}.png", dpi=150)\n    plt.close(fig)\n    (out / f"{name}.json").write_text(json.dumps(summary, indent=2, default=float), encoding="utf-8")\n    return summary\n', 'meta_brain/mario_trainable.py': '"""Full-connectome recurrent PPO, with bounded BPTT and restartable Colab runs.\n\nV2 remains the frozen-reservoir baseline. V3 trains a visual encoder, cell-type\ninput/recurrent gains, biases and time constants, and nonlinear motor heads.\nThe signed edges and topology stay fixed; there is no image-to-action bypass.\nResume restores training, not the emulator\'s exact in-flight trajectories.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Callable\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom .graph import ConnectomeArrays\nfrom .mario_rl import MarioRunner, VisualInput, episode_key, save_video, summarize\nfrom .sparse_ops import SparseStructure, sparse_propagate\n\n\n@dataclass\nclass FullBrainConfig:\n    seed: int = 123\n    vision: str = "real"\n    n_envs: int = 4\n    rollout: int = 128\n    bptt: int = 16\n    micro_steps: int = 4\n    updates: int = 10_000\n    epochs: int = 3\n    lr: float = 1e-4\n    gamma: float = 0.99\n    gae_lambda: float = 0.95\n    clip: float = 0.2\n    entropy: float = 0.02\n    value_coef: float = 0.5\n    reward_scale: float = 0.1\n    max_grad_norm: float = 0.5\n    target_kl: float = 0.03\n    frame_skip: int = 4\n    stuck_steps: int = 90\n    max_steps: int = 2_000\n    noop_max: int = 30\n    checkpoint_every: int = 10\n    eval_every: int = 50\n    validation_episodes: int = 3\n    final_episodes: int = 20\n    max_hours: float = 3.0\n    latent: int = 64\n    hidden: int = 128\n\n    def validate(self):\n        for key in ("n_envs", "rollout", "bptt", "micro_steps", "updates", "epochs", "frame_skip",\n                    "checkpoint_every", "eval_every", "validation_episodes", "final_episodes"):\n            if getattr(self, key) <= 0:\n                raise ValueError(f"{key} must be positive")\n        if self.rollout % self.bptt:\n            raise ValueError("rollout must be divisible by bptt")\n        if self.vision not in ("real", "zero", "clock"):\n            raise ValueError("vision must be real, zero or clock")\n\n\ndef graph_fingerprint(graph: ConnectomeArrays) -> str:\n    """Content identity; a same-size but shuffled graph must not load a checkpoint."""\n    h = hashlib.sha256()\n    for a in (graph.body_id, graph.pre, graph.post, graph.count, graph.nt_code,\n              graph.cell_type, graph.superclass):\n        a = np.ascontiguousarray(a)\n        h.update(str(a.dtype).encode())\n        h.update(memoryview(a).cast("B"))\n    h.update(json.dumps([graph.type_names, graph.superclass_names]).encode())\n    return h.hexdigest()\n\n\ndef reservoir_weights(graph: ConnectomeArrays) -> np.ndarray:\n    w = np.log1p(graph.count).astype(np.float32)\n    energy = np.bincount(graph.post, weights=w * w, minlength=graph.n)\n    w /= np.sqrt(np.maximum(energy[graph.post], 1e-6)).astype(np.float32)\n    return w * graph.sign()[graph.pre]\n\n\nclass TrainableBrain(nn.Module):\n    """All visual information must traverse the sparse biological graph.\n\n    Cell-type parameters modulate neuron dynamics rather than creating E-sized\n    trainable weights. The N x B activations dominate the BPTT tape, not E x B.\n    The fixed latent-to-sensory basis is compact (64 x sensory, not 2048 x sensory).\n    It is not claimed to be a biological retinotopic map.\n    """\n    def __init__(self, structure: SparseStructure, weights, sensory, motor, cell_type,\n                 n_actions: int, cfg: FullBrainConfig):\n        super().__init__()\n        cfg.validate()\n        self.structure, self.cfg, self.n_actions = structure, cfg, n_actions\n        device = structure.device\n        # Constant graph tensors are rebuilt from fingerprinted data on resume.\n        for name, a, dtype in (("weights", weights, torch.float32), ("sensory", sensory, torch.long),\n                               ("motor", motor, torch.long), ("types", cell_type, torch.long)):\n            self.register_buffer(name, torch.as_tensor(a, dtype=dtype, device=device), persistent=False)\n        if not len(self.sensory) or not len(self.motor):\n            raise ValueError("The full graph needs visual sensory and motor neurons")\n        n_types = int(self.types.max()) + 1\n        gen = torch.Generator().manual_seed(cfg.seed)\n        proj = torch.randn(cfg.latent, len(self.sensory), generator=gen) / math.sqrt(cfg.latent)\n        self.register_buffer("projection", proj.to(device), persistent=False)\n        self.encoder = nn.Sequential(\n            nn.Conv2d(2, 16, 5, stride=2, padding=2), nn.ELU(),\n            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ELU(),\n            nn.Flatten(), nn.Linear(32 * 8 * 8, cfg.latent), nn.Tanh())\n        # 0 < recurrence < .9; .2 < decay < .95; .25 < input gain < 4.\n        self.recurrent = nn.Parameter(torch.full((n_types,), math.log(0.5 / 0.4)))\n        self.decay = nn.Parameter(torch.full((n_types,), math.log(0.5 / 0.25)))  # initial .7\n        self.input_scale = nn.Parameter(torch.full((n_types,), math.log(0.75 / 3.0)))\n        self.bias = nn.Parameter(torch.zeros(n_types))\n        self.motor_norm = nn.LayerNorm(len(self.motor), eps=1e-8)\n        self.head = nn.Sequential(nn.Linear(len(self.motor), cfg.hidden), nn.Tanh())\n        self.pi, self.value = nn.Linear(cfg.hidden, n_actions), nn.Linear(cfg.hidden, 1)\n        nn.init.orthogonal_(self.pi.weight, gain=0.01)\n        nn.init.zeros_(self.pi.bias)\n        self.to(device)\n\n    @property\n    def device(self):\n        return self.weights.device\n\n    def initial_state(self, batch):\n        return torch.zeros(self.structure.n, batch, device=self.device)\n\n    def forward(self, images, state):\n        code = self.encoder(images)\n        if self.cfg.vision == "zero":\n            code = code * 0  # blind control; learned intrinsic dynamics may still form a clock\n        scale = .25 + 3.75 * torch.sigmoid(self.input_scale[self.types[self.sensory]])\n        drive = torch.zeros_like(state).index_add(0, self.sensory, (code @ self.projection).t() * scale[:, None])\n        gain = .9 * torch.sigmoid(self.recurrent[self.types])[:, None]\n        decay = .2 + .75 * torch.sigmoid(self.decay[self.types])[:, None]\n        bias = .1 * torch.tanh(self.bias[self.types])[:, None]\n        for _ in range(self.cfg.micro_steps):\n            message = sparse_propagate(state, self.weights, self.structure)\n            state = decay * state + (1 - decay) * torch.tanh(gain * message + drive + bias)\n        motor = state.index_select(0, self.motor).t()\n        h = self.head(self.motor_norm(motor * 10))\n        return self.pi(h), self.value(h).squeeze(-1), state\n\n\nclass RecurrentCollector:\n    def __init__(self, model, factory, cfg, seed_offset=0):\n        self.model, self.cfg = model, cfg\n        self.runners = [MarioRunner(factory(), cfg.frame_skip, cfg.noop_max, cfg.stuck_steps,\n                                    cfg.max_steps, cfg.seed + seed_offset + i) for i in range(cfg.n_envs)]\n        self.vision = VisualInput(cfg.vision, seed=cfg.seed)\n        self.prev = [None] * cfg.n_envs\n        self.images = np.stack([self.encode(i, r.reset()) for i, r in enumerate(self.runners)])\n        self.state = model.initial_state(cfg.n_envs)\n        self.reset_mask = np.zeros(cfg.n_envs, np.float32)\n\n    def encode(self, i, obs):\n        code, self.prev[i] = self.vision(obs, self.prev[i], self.runners[i].steps)\n        return code.reshape(2, 32, 32)\n\n    def close(self):\n        for r in self.runners:\n            r.env.close()\n\n    @torch.no_grad()\n    def collect(self):\n        cfg, device = self.cfg, self.model.device\n        data = {k: [] for k in ("images", "resets", "actions", "logp", "values", "rewards", "dones", "bootstrap")}\n        boundaries, episodes = [], []\n        for t in range(cfg.rollout):\n            if t % cfg.bptt == 0:\n                boundaries.append(self.state.detach().clone())\n            images = torch.as_tensor(self.images, device=device)\n            masks = torch.as_tensor(self.reset_mask, device=device)\n            state = self.state * (1 - masks)[None, :]\n            logits, values, next_state = self.model(images, state)\n            dist = torch.distributions.Categorical(logits=logits)\n            actions = dist.sample()\n            next_images, rewards, dones, terminated = [], [], [], []\n            terminal_images, cutoffs = [], []\n            for i, (runner, action) in enumerate(zip(self.runners, actions.tolist())):\n                obs, reward, done, info = runner.step(action)\n                encoded = self.encode(i, obs)\n                terminal_images.append(encoded)\n                rewards.append(reward * cfg.reward_scale)\n                dones.append(done)\n                terminated.append(bool(info.get("terminated", done and not info["truncated"])))\n                if done:\n                    episodes.append(runner.episode_summary())\n                    self.prev[i] = None\n                    encoded = self.encode(i, runner.reset())\n                next_images.append(encoded)\n                cutoffs.append(bool(done and not terminated[-1]))\n            # Reuse the next step\'s value for continuing transitions; evaluate only\n            # time-limit/stuck terminal observations separately (never the reset frame).\n            bootstrap = torch.zeros(cfg.n_envs, device=device)\n            ids = np.flatnonzero(cutoffs)\n            if len(ids):\n                _, v, _ = self.model(torch.as_tensor(np.stack([terminal_images[i] for i in ids]), device=device),\n                                     next_state[:, ids])\n                bootstrap[torch.as_tensor(ids, device=device)] = v\n            for key, value in (("images", images), ("resets", masks), ("actions", actions),\n                               ("logp", dist.log_prob(actions)), ("values", values),\n                               ("rewards", torch.tensor(rewards, device=device)),\n                               ("dones", torch.tensor(dones, device=device)), ("bootstrap", bootstrap)):\n                data[key].append(value)\n            self.images = np.stack(next_images)\n            self.state, self.reset_mask = next_state, np.asarray(dones, np.float32)\n        _, last, _ = self.model(torch.as_tensor(self.images, device=device),\n                                self.state * (1 - torch.as_tensor(self.reset_mask, device=device))[None, :])\n        data = {k: torch.stack(v) for k, v in data.items()}\n        nxt = torch.cat([data["values"][1:], last[None]], dim=0)\n        nxt = torch.where(data["dones"], data["bootstrap"], nxt)\n        data["advantages"], data["returns"] = generalized_advantages(\n            data["rewards"], data["values"], nxt, data["dones"], cfg.gamma, cfg.gae_lambda)\n        data["states"], data["episodes"] = boundaries, episodes\n        return data\n\n\ndef generalized_advantages(rewards, values, next_values, dones, gamma, lam):\n    """next_values is zero for deaths, terminal value for time limits, next value otherwise."""\n    adv, running = torch.zeros_like(values), torch.zeros_like(values[0])\n    for t in reversed(range(len(rewards))):\n        delta = rewards[t] + gamma * next_values[t] - values[t]\n        running = delta + gamma * lam * (~dones[t]).float() * running\n        adv[t] = running\n    return adv, adv + values\n\n\ndef ppo_update(model, optimizer, data, cfg):\n    """Shuffle contiguous sequences, never timesteps; differentiate through the brain.\n\n    Boundary states are detached rollout states (truncated recurrent PPO), and\n    within-sequence episode resets also cut the recurrent gradient.\n    """\n    adv = data["advantages"]\n    adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)\n    stats = []\n    for _ in range(cfg.epochs):\n        for block in torch.randperm(cfg.rollout // cfg.bptt).tolist():\n            start = block * cfg.bptt\n            state = data["states"][block].detach()\n            logits, vals = [], []\n            for t in range(start, start + cfg.bptt):\n                state = state * (1 - data["resets"][t])[None, :]\n                pi, v, state = model(data["images"][t], state)\n                logits.append(pi)\n                vals.append(v)\n            sl = slice(start, start + cfg.bptt)\n            dist = torch.distributions.Categorical(logits=torch.stack(logits))\n            logp = dist.log_prob(data["actions"][sl])\n            logratio = logp - data["logp"][sl]\n            ratio = logratio.exp()\n            pg = -torch.minimum(ratio * adv[sl], ratio.clamp(1 - cfg.clip, 1 + cfg.clip) * adv[sl]).mean()\n            v = torch.stack(vals)\n            vclip = data["values"][sl] + (v - data["values"][sl]).clamp(-cfg.clip, cfg.clip)\n            vloss = .5 * torch.maximum((v - data["returns"][sl]).square(),\n                                      (vclip - data["returns"][sl]).square()).mean()\n            ent = dist.entropy().mean()\n            loss = pg + cfg.value_coef * vloss - cfg.entropy * ent\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Nonfinite PPO loss; last complete checkpoint is preserved")\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm, error_if_nonfinite=True)\n            kl = float(((ratio - 1) - logratio).mean().detach())\n            # Stop before applying an update already outside the KL trust region.\n            if kl > cfg.target_kl:\n                optimizer.zero_grad(set_to_none=True)\n                return _mean_stats(stats)\n            optimizer.step()\n            stats.append(dict(loss=float(loss.detach()), entropy=float(ent.detach()), kl=kl, grad_norm=float(norm)))\n    return _mean_stats(stats)\n\n\ndef _mean_stats(rows):\n    return {k: float(np.mean([r[k] for r in rows])) for k in ("loss", "entropy", "kl", "grad_norm")} if rows else {}\n\n\n@torch.no_grad()\ndef evaluate_brain(model, factory, cfg, episodes, seed, greedy=True, record=False, vision_mode=None):\n    """Fresh seeded starts for matched before/after and held-out final evaluation."""\n    runner = MarioRunner(factory(), cfg.frame_skip, cfg.noop_max, cfg.stuck_steps, cfg.max_steps,\n                         seed, record_stride=cfg.frame_skip)\n    vision = VisualInput(vision_mode or cfg.vision, seed=cfg.seed)\n    generator = torch.Generator().manual_seed(seed)\n    rows, best, frames = [], None, []\n    try:\n        for _ in range(episodes):\n            obs, prev = runner.reset(record=record), None\n            state, done = model.initial_state(1), False\n            while not done:\n                code, prev = vision(obs, prev, runner.steps)\n                image = torch.as_tensor(code.reshape(1, 2, 32, 32), device=model.device)\n                logits, _, state = model(image, state)\n                action = (int(logits.argmax(-1)) if greedy else\n                          int(torch.multinomial(logits.softmax(-1).cpu(), 1, generator=generator)))\n                obs, _, done, _ = runner.step(action)\n            row = runner.episode_summary()\n            rows.append(row)\n            if best is None or episode_key(row) > episode_key(best):\n                best, frames = row, runner.frames\n    finally:\n        runner.env.close()\n    return summarize(rows), frames\n\n\ndef atomic_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_checkpoint(path, model, optimizer, cfg, graph_id, update, history, best_key):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    state = {"format": 3, "config": asdict(cfg), "graph_id": graph_id, "update": update,\n             "model": {k: v.detach().cpu() for k, v in model.state_dict().items()},\n             "optimizer": optimizer.state_dict(), "history": history, "best_key": list(best_key),\n             "torch_rng": torch.get_rng_state(),\n             "cuda_rng": torch.cuda.get_rng_state_all() if model.device.type == "cuda" else [],\n             "resume_semantics": "weights_optimizer_counters_rng; fresh emulator episodes"}\n    torch.save(state, tmp)\n    tmp.replace(path)\n\n\ndef load_checkpoint(path, model, optimizer, cfg, graph_id):\n    ck = torch.load(path, map_location="cpu", weights_only=True)\n    if ck.get("format") != 3 or ck["graph_id"] != graph_id:\n        raise ValueError("Checkpoint belongs to another graph or format; choose a new run folder")\n    mutable = {"updates", "max_hours", "checkpoint_every", "eval_every", "validation_episodes", "final_episodes"}\n    for k, v in asdict(cfg).items():\n        if k not in mutable and ck["config"].get(k) != v:\n            raise ValueError(f"Resume config mismatch: {k}; use the original config or a new run folder")\n    model.load_state_dict(ck["model"])\n    if optimizer is not None:\n        optimizer.load_state_dict(ck["optimizer"])\n        for state in optimizer.state.values():\n            for k, value in state.items():\n                if torch.is_tensor(value):\n                    state[k] = value.to(model.device)\n    torch.set_rng_state(ck["torch_rng"])\n    if model.device.type == "cuda" and ck["cuda_rng"]:\n        torch.cuda.set_rng_state_all(ck["cuda_rng"])\n    return ck\n\n\ndef preflight(model, cfg):\n    """Actual forward/backward on the loaded full graph, before a long run."""\n    if model.device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(model.device)\n    state = model.initial_state(cfg.n_envs)\n    loss = torch.zeros((), device=model.device)\n    for _ in range(cfg.bptt):\n        logits, v, state = model(torch.randn(cfg.n_envs, 2, 32, 32, device=model.device), state)\n        loss = loss + logits.square().mean() + v.square().mean()\n    loss.backward()\n    grads = {name: float(p.grad.norm()) for name, p in model.named_parameters() if p.grad is not None}\n    if not all(math.isfinite(g) for g in grads.values()):\n        raise FloatingPointError("Preflight produced a nonfinite gradient")\n    if not any(g > 0 for k, g in grads.items() if k.startswith("encoder")):\n        raise RuntimeError("No visual gradient reaches the action head through this graph/BPTT window")\n    # Include Adam\'s two moment buffers in the measured budget, without changing weights.\n    probe_optimizer = torch.optim.Adam(model.parameters(), lr=0)\n    probe_optimizer.step()\n    model.zero_grad(set_to_none=True)\n    return {"trainable_parameters": sum(p.numel() for p in model.parameters()),\n            "encoder_gradient_norm": sum(g for k, g in grads.items() if k.startswith("encoder")),\n            "peak_gpu_gib": (torch.cuda.max_memory_allocated(model.device) / 2**30\n                             if model.device.type == "cuda" else None),\n            "n_envs": cfg.n_envs, "bptt": cfg.bptt, "micro_steps": cfg.micro_steps}\n\n\ndef run_full_brain(model, factory: Callable, cfg, out_dir, graph_id, resume=True, log=print):\n    """Persistent last/best checkpoints, bounded session time, independent final evaluation."""\n    cfg.validate()\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)\n    latest, best_path = out / "latest.pt", out / "best.pt"\n    history, first, best_key = [], 0, (-1.0, -1.0)\n    if latest.exists() and not resume:\n        raise FileExistsError("Run already exists; resume it or choose another folder")\n    if resume and latest.exists():\n        ck = load_checkpoint(latest, model, optimizer, cfg, graph_id)\n        first, history, best_key = ck["update"], ck["history"], tuple(ck["best_key"])\n        log(f"Resuming update {first}; emulator episodes restart.")\n    elif not (out / "pre_eval.json").exists():\n        before, _ = evaluate_brain(model, factory, cfg, cfg.validation_episodes, cfg.seed + 100_000)\n        atomic_json(out / "pre_eval.json", before)\n    # An update=0 checkpoint makes the first session recoverable too.\n    save_checkpoint(latest, model, optimizer, cfg, graph_id, first, history, best_key)\n    collector = RecurrentCollector(model, factory, cfg, seed_offset=first * cfg.n_envs)\n    started, completed, stop_reason = time.monotonic(), first, "update_target"\n    try:\n        for update in range(first, cfg.updates):\n            if cfg.max_hours > 0 and time.monotonic() - started >= cfg.max_hours * 3600:\n                stop_reason = "session_time_budget"\n                break\n            t0 = time.monotonic()\n            data = collector.collect()\n            stats = ppo_update(model, optimizer, data, cfg)\n            episodes = data.pop("episodes")\n            del data\n            row = {"update": update + 1, "agent_steps": (update + 1) * cfg.rollout * cfg.n_envs,\n                   "episodes": len(episodes), "flags": sum(e["flag_get"] for e in episodes),\n                   "mean_max_x": float(np.mean([e["max_x"] for e in episodes])) if episodes else None,\n                   "best_max_x": max((e["max_x"] for e in episodes), default=None),\n                   "seconds": round(time.monotonic() - t0, 3), **stats}\n            history.append(row)\n            completed = update + 1\n            if completed % cfg.checkpoint_every == 0:\n                save_checkpoint(latest, model, optimizer, cfg, graph_id, completed, history, best_key)\n                atomic_json(out / "progress.json", {"update": completed, "latest": row})\n            if completed % cfg.eval_every == 0:\n                result, _ = evaluate_brain(model, factory, cfg, cfg.validation_episodes, cfg.seed + 100_000)\n                key = (result["success_rate"], result["max_x_mean"])\n                if key > best_key:\n                    best_key = key\n                    save_checkpoint(best_path, model, optimizer, cfg, graph_id, completed, history, best_key)\n                    atomic_json(out / "best_validation.json", result)\n                # Persist best_key in latest even if eval/checkpoint periods differ.\n                save_checkpoint(latest, model, optimizer, cfg, graph_id, completed, history, best_key)\n            if completed == 1 or completed % cfg.checkpoint_every == 0:\n                log(f"update {completed}/{cfg.updates} steps={row[\'agent_steps\']:,} "\n                    f"mean_x={row[\'mean_max_x\']} flags={row[\'flags\']} entropy={row.get(\'entropy\', 0):.3f}")\n        save_checkpoint(latest, model, optimizer, cfg, graph_id, completed, history, best_key)\n    except KeyboardInterrupt:\n        log("Interrupted. Resume from the last complete checkpoint; partial updates were not saved.")\n        raise\n    finally:\n        collector.close()\n    # Final RNG seeds differ from validation. Finite no-op offsets may overlap;\n    # this is evaluation on the same stage, not held-out-level generalization.\n    selected = best_path if best_path.exists() else latest\n    selected_ck = load_checkpoint(selected, model, None, cfg, graph_id)\n    greedy, frames = evaluate_brain(model, factory, cfg, cfg.final_episodes, cfg.seed + 200_000, record=True)\n    sampled, sampled_frames = evaluate_brain(model, factory, cfg, cfg.final_episodes,\n                                            cfg.seed + 200_000, greedy=False, record=True)\n    codec = save_video(frames, out / "post_greedy.mp4", fps=max(1, round(60 / cfg.frame_skip)))\n    save_video(sampled_frames, out / "post_sampled.mp4", fps=max(1, round(60 / cfg.frame_skip)))\n    results = {"experiment": "Meta-Brain V3 trainable full-connectome PPO", "graph_id": graph_id,\n               "config": asdict(cfg), "completed_updates": completed, "selected_update": selected_ck["update"],\n               "stop_reason": stop_reason, "post_eval": greedy, "post_eval_sampled": sampled,\n               "selected_checkpoint": selected.name, "video_codec": codec,\n               "trainable": "CNN vision; cell-type input/recurrent gains, decay, bias; nonlinear motor heads",\n               "fixed": "signed synapse counts, topology and compact sensory projection basis; no visual bypass",\n               "claim_status": ("DEMONSTRATED_SUCCESS" if greedy["all_success_no_death"]\n                                else "RUN_COMPLETED_BUT_SUCCESS_NOT_DEMONSTRATED"),\n               "resume_semantics": "training resumes; in-flight emulator trajectories restart",\n               "history": history}\n    atomic_json(out / "results.json", results)\n    return results\n', 'meta_brain/colab_mario.py': '"""Shared Colab/CLI preparation; no full dataset or GPU is needed to import it."""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport platform\nimport shutil\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .graph import load_connectome\nfrom .mario_rl import make_mario_env\nfrom .mario_trainable import (FullBrainConfig, TrainableBrain, atomic_json, graph_fingerprint,\n                              preflight, reservoir_weights, run_full_brain)\nfrom .sparse_ops import SparseStructure\n\n\nDATA_FILES = (\n    "body-annotations-male-cns-v1.0-minconf-0.5.feather",\n    "body-neurotransmitters-male-cns-v1.0.feather",\n    "connectome-weights-male-cns-v1.0-minconf-0.5.feather",\n)\nDATA_BASE = "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome"\n\n\ndef fetch_data(data_dir, persistent_dir=None, log=print):\n    """Stream each download, check length and Feather readability before atomic promotion."""\n    import requests\n    import pyarrow.feather as feather\n\n    local = Path(data_dir)\n    local.mkdir(parents=True, exist_ok=True)\n    persistent = Path(persistent_dir) if persistent_dir is not None else None\n    if persistent is not None:\n        persistent.mkdir(parents=True, exist_ok=True)\n    for name in DATA_FILES:\n        path = local / name\n        if path.exists():\n            # memory_map avoids loading the ~1 GB file just to verify its schema.\n            feather.read_table(path, columns=[], memory_map=True)\n            continue\n        tmp = path.with_name(path.name + ".part")\n        if persistent is not None and (persistent / name).exists():\n            log(f"Copying {name} from Drive")\n            shutil.copyfile(persistent / name, tmp)\n        else:\n            log(f"Downloading {name}")\n            with requests.get(f"{DATA_BASE}/{name}", stream=True, timeout=(30, 180)) as response:\n                response.raise_for_status()\n                expected = int(response.headers.get("content-length", 0))\n                written = 0\n                with tmp.open("wb") as f:\n                    for chunk in response.iter_content(8 << 20):\n                        written += f.write(chunk)\n                if expected and written != expected:\n                    raise IOError(f"Incomplete download: {name}: {written}/{expected} bytes")\n        feather.read_table(tmp, columns=[], memory_map=True)\n        tmp.replace(path)\n        if persistent is not None and not (persistent / name).exists():\n            dest_tmp = persistent / (name + ".part")\n            shutil.copyfile(path, dest_tmp)\n            dest_tmp.replace(persistent / name)\n\n\ndef runtime_info(device):\n    import importlib.metadata\n    return {"python": platform.python_version(), "torch": str(torch.__version__),\n            "device": str(device), "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,\n            "gpu_gib": torch.cuda.get_device_properties(device).total_memory / 2**30 if device.type == "cuda" else None,\n            "packages": {p: importlib.metadata.version(p) for p in\n                         ("numpy", "pyarrow", "gymnasium", "gym-super-mario-bros", "nes-py")}}\n\n\ndef prepare_brain(data_dir, cache_dir, out_dir, cfg, device="cuda", min_neurons=100_000,\n                  min_edges=5_000_000, auto_memory=True, log=print):\n    device = torch.device(device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("Select a GPU runtime in Colab, then run all cells again")\n    if shutil.which("ffmpeg") is None:\n        raise RuntimeError("ffmpeg is required for playable H.264 videos; run the installation cell")\n    # The real emulator is tested BEFORE the expensive graph build.\n    env, actions = make_mario_env()\n    try:\n        env.reset()\n        env.step(1)\n    finally:\n        env.close()\n    graph = load_connectome(data_dir, cache_dir=cache_dir)\n    if graph.n < min_neurons or graph.e < min_edges:\n        raise RuntimeError(f"Full-brain guard failed: {graph.n:,} neurons and {graph.e:,} edges")\n    visual = np.flatnonzero(graph.superclass_mask(("ol_sensory", "olsn")))\n    motor = graph.motor_indices()\n    if not len(visual):\n        raise ValueError("No optic sensory cells found; refusing to inject images into nonvisual senses")\n    graph_id = graph_fingerprint(graph)\n    info = {"neurons": graph.n, "edges": graph.e, "visual_sensory": len(visual), "motor": len(motor),\n            "graph_id": graph_id, "load_report": asdict(graph.report)}\n    log(graph.report.summary())\n    # Existing runs keep their original batch, BPTT and architecture across GPU changes.\n    latest = Path(out_dir) / "latest.pt"\n    if latest.exists():\n        saved = torch.load(latest, map_location="cpu", weights_only=True)\n        hours, updates, final_episodes = cfg.max_hours, cfg.updates, cfg.final_episodes\n        requested_vision, requested_seed = cfg.vision, cfg.seed\n        cfg = FullBrainConfig(**saved["config"])\n        if cfg.vision != requested_vision or cfg.seed != requested_seed:\n            raise ValueError("Run folder contains another vision/seed; choose a new run name")\n        cfg.max_hours, cfg.updates, cfg.final_episodes = hours, updates, final_episodes\n        auto_memory = False\n    weights = reservoir_weights(graph)\n    structure = SparseStructure.from_edges(graph.pre, graph.post, graph.n, device=device, index_dtype=torch.int32)\n    types = graph.cell_type.copy()\n    del graph\n    gc.collect()\n    while True:\n        torch.manual_seed(cfg.seed)\n        model = None\n        try:\n            model = TrainableBrain(structure, weights, visual, motor, types, len(actions), cfg)\n            check = preflight(model, cfg)\n            break\n        except torch.cuda.OutOfMemoryError:\n            del model\n            gc.collect()\n            torch.cuda.empty_cache()\n            if not auto_memory or (cfg.n_envs == 1 and cfg.bptt <= 4):\n                raise RuntimeError("Full graph/BPTT does not fit this GPU. Use more VRAM, or a new run with smaller batch/BPTT") from None\n            if cfg.n_envs > 1:\n                cfg.n_envs = max(1, cfg.n_envs // 2)\n            else:\n                cfg.bptt = max(4, cfg.bptt // 2)\n            log(f"Retrying preflight with {cfg.n_envs} environments and BPTT={cfg.bptt}")\n    atomic_json(Path(out_dir) / "preflight.json", {"graph": info, "memory_and_gradients": check, "config": asdict(cfg)})\n    log(f"Preflight OK: {json.dumps(check)}")\n    return model, cfg, info\n\n\ndef launch(data_dir, cache_dir, out_dir, cfg=None, device="cuda", min_neurons=100_000,\n           min_edges=5_000_000, auto_memory=True, log=print):\n    cfg = cfg or FullBrainConfig()\n    model, cfg, info = prepare_brain(data_dir, cache_dir, out_dir, cfg, device, min_neurons, min_edges, auto_memory, log)\n    try:\n        results = run_full_brain(model, lambda: make_mario_env()[0], cfg, out_dir, info["graph_id"], log=log)\n        results["graph"] = info\n        atomic_json(Path(out_dir) / "results.json", results)\n        return results\n    finally:\n        del model\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n', 'meta_brain/__init__.py': '"""Self-contained Meta-Brain Colab runtime."""\n'}
for relative, content in SOURCE_FILES.items():
    path = RUNTIME_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
sys.path.insert(0, str(RUNTIME_DIR))
# Re-running all cells uses the newly embedded code, even after edits.
for name in list(sys.modules):
    if name == 'meta_brain' or name.startswith('meta_brain.'):
        del sys.modules[name]
print('Código embutido:', SOURCE_DIGEST[:12])


## 4. Dados, auditoria e teste de memória

O pré-teste calcula gradientes no grafo completo. Se faltar memória, reduz o lote; a escala do conectoma é preservada. Em uma retomada, conserva a configuração original. O primeiro carregamento exige mais RAM que os seguintes.

In [ ]:
from meta_brain.colab_mario import fetch_data, prepare_brain, runtime_info
from meta_brain.mario_trainable import FullBrainConfig, atomic_json, run_full_brain
from meta_brain.mario_rl import make_mario_env

cfg = FullBrainConfig(seed=SEED, vision=VISION, updates=TARGET_UPDATES, max_hours=SESSION_HOURS,
                      final_episodes=FINAL_EPISODES, n_envs=N_ENVS, bptt=BPTT, micro_steps=MICRO_STEPS)
if OFFLINE:
    cfg.n_envs = int(os.environ.get('MB_MARIO_ENVS', 2))
    cfg.rollout, cfg.bptt = 8, 4
    cfg.micro_steps = int(os.environ.get('MB_MARIO_MICRO_STEPS', 2))
    cfg.updates = int(os.environ.get('MB_MARIO_UPDATES', 1))
    cfg.epochs, cfg.max_steps, cfg.noop_max = 1, 12, 0
    cfg.eval_every = cfg.checkpoint_every = 1
    cfg.validation_episodes = cfg.final_episodes = 1
    cfg.max_hours = 0
cfg.validate()
if not OFFLINE:
    # Fail fast on the emulator before downloading 1.2 GB of connectome data.
    probe, _ = make_mario_env()
    try:
        observation, info = probe.reset(seed=SEED)
        probe.step(1)
        print('Emulador real OK:', observation.shape)
    finally:
        probe.close()
    fetch_data(DATA_DIR, PERSISTENT_DATA)
    atomic_json(RUN_DIR / 'runtime.json', runtime_info(torch.device(DEVICE)))
MIN_NEURONS = int(os.environ.get('MB_MIN_NEURONS', 100000))
MIN_EDGES = int(os.environ.get('MB_MIN_EDGES', 5000000))
model, cfg, GRAPH_INFO = prepare_brain(DATA_DIR, CACHE_DIR, RUN_DIR, cfg, DEVICE,
                                      MIN_NEURONS, MIN_EDGES, auto_memory=True)
print('Configuração efetiva:', cfg)
print('Grafo completo:', GRAPH_INFO['neurons'], 'neurônios;', GRAPH_INFO['edges'], 'arestas')


## 5. Treinar ou continuar

Para continuar na próxima sessão, mantenha o nome e execute todas as células. Aumente `TARGET_UPDATES` para estender o treino. A avaliação escolhe o melhor checkpoint em inícios de validação e testa em outros inícios. Os controles `zero` e `clock` usam pastas e treinamentos independentes; compare o mesmo número de updates, lote e sementes.

In [ ]:
# Salva latest.pt a cada 10 updates e best.pt nas avaliações de validação.
# Interromper/perder a sessão mantém o último checkpoint completo no Drive.
RESULTS = run_full_brain(model, lambda: make_mario_env()[0], cfg, RUN_DIR, GRAPH_INFO['graph_id'])
RESULTS['graph'] = GRAPH_INFO
atomic_json(RUN_DIR / 'results.json', RESULTS)
print('Resultado:', RESULTS['claim_status'])
print('Vitórias (determinístico):', RESULTS['post_eval']['success_rate'])
print('Vitórias (amostrado):', RESULTS['post_eval_sampled']['success_rate'])


## 6. Resultados e vídeos

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Video, display
history = RESULTS['history']
if history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([r['agent_steps'] for r in history], [r['mean_max_x'] for r in history])
    axes[0].set_title('Progresso médio nos episódios encerrados')
    axes[0].set_xlabel('Decisões acumuladas'); axes[0].set_ylabel('Maior posição x')
    axes[1].plot([r['agent_steps'] for r in history], [r.get('entropy') for r in history])
    axes[1].set_title('Exploração da política'); axes[1].set_xlabel('Decisões acumuladas')
    fig.tight_layout(); fig.savefig(RUN_DIR / 'training.png', dpi=140); plt.show()
for name in ('post_greedy.mp4', 'post_sampled.mp4'):
    print(name, '— melhor episódio, não a média')
    display(Video(str(RUN_DIR / name), embed=True))
print('Arquivos:', *sorted(p.name for p in RUN_DIR.iterdir()), sep='\n  ')


## Interpretação e retomada

`latest.pt`: continuar o treino. `best.pt`: melhor checkpoint de validação. `preflight.json`: escala real, memória e gradiente visual. `results.json`: avaliações finais completas. Os vídeos mostram apenas o melhor episódio. Uma taxa de sucesso zero é uma execução válida sem vitória demonstrada.

Após desconexão, selecione GPU novamente e execute tudo com os mesmos parâmetros. Não apague a pasta do Drive. Caso a GPU da nova sessão seja menor e a configuração salva não caiba, obtenha uma GPU com mais memória; não altere silenciosamente a configuração de um experimento em andamento.

A V3 é uma nova hipótese de arquitetura. Compare vários seeds e controles antes de atribuir resultados à topologia biológica. Não equivale a uma reprodução biológica completa, nem a um treino independente de cada sinapse.